In [1]:
import subprocess, base64, os, glob, shutil
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/Rosierobber/Temporal-Graph-ML-Project.git"
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/Temporal-Graph-ML-Project"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()

def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", "-c", f"http.extraheader={hdr}", *args],
                       cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    return r.returncode

def save_and_push(msg, paths=("notes", "results", "logs", "README.md")):
    git("add", *paths)
    git("status", "--short")
    git("commit", "-m", msg)
    git("pull", "--rebase", "origin", "main")
    git("push", "origin", "main")

if not os.path.exists(REPO_DIR):
    git("clone", REPO_URL, cwd=WORK)
git("config", "user.name", "Rabia Habib")
git("config", "user.email", "rabihabib0411@gmail.com")
git("pull", "--rebase", "origin", "main")
git("log", "--oneline", "-3")

Cloning into 'Temporal-Graph-ML-Project'...



Already up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD

99c3109 Fig 3: sensitivity to window ratio
3d36290 R-EdgeBank on tgbl-review
e834285 EdgeBank on tgbl-review



0

In [3]:
!pip install -q py-tgb torch_geometric
import tgb
print("ok")

ok


In [15]:
BASE_URL    = "https://github.com/fpour/TGB_Baselines.git"
BASE_COMMIT = "d2022fae4a3311185f611bab733e267938251f42"
BASE        = f"{REPO_DIR}/third_party/TGB_Baselines"

if not os.path.exists(f"{BASE}/train_tgb_lpp_original.py"):
    os.makedirs(os.path.dirname(BASE), exist_ok=True)
    subprocess.run(["git", "clone", "-q", BASE_URL, BASE], check=True)
    subprocess.run(["git", "checkout", "-q", BASE_COMMIT], cwd=BASE, check=True)

actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=BASE, text=True).strip()
print("TGB_Baselines commit:", actual)
assert actual == BASE_COMMIT, "Wrong TGB_Baselines commit"

# make sure it stays out of your repo
gi = f"{REPO_DIR}/.gitignore"
lines = open(gi).read().splitlines() if os.path.exists(gi) else []
if "third_party/TGB_Baselines/" not in lines:
    open(gi, "a").write("third_party/TGB_Baselines/\n")

TGB_Baselines commit: d2022fae4a3311185f611bab733e267938251f42


In [4]:
import glob, shutil, os, json, statistics as st

TGB = f"{REPO_DIR}/third_party/TGB"
SAVED = f"{TGB}/examples/linkproppred/tgbl-wiki/saved_results"
os.makedirs(f"{REPO_DIR}/results/edgebank_seeds", exist_ok=True)

for mode, flag, tag in [("unlimited", "", "inf"),
                        ("fixed_time_window", "--mem_mode fixed_time_window", "tw")]:
    for s in [1, 2, 3]:
        log_path = f"{REPO_DIR}/logs/edgebank_{tag}_wiki_seed{s}.txt"
        if os.path.exists(log_path):
            print(f"EdgeBank {tag} seed {s} already done, skipping")
            continue
        !cd {TGB} && PYTHONPATH=. python examples/linkproppred/tgbl-wiki/edgebank.py --data tgbl-wiki --seed {s} {flag} 2>&1 | tee {log_path}
        if "Test: mrr" not in open(log_path).read():
            print(f"EdgeBank {tag} seed {s} did not finish. Check the log. Stopping.")
            break
        for f in glob.glob(f"{SAVED}/EdgeBank_{mode}*"):
            shutil.copy(f, f"{REPO_DIR}/results/edgebank_seeds/seed{s}_{os.path.basename(f)}")

/usr/local/lib/python3.13/dist-packages/clint/textui/colored.py:118: SyntaxWarning: invalid escape sequence '\$'
  strip = re.compile("([^-_a-zA-Z0-9!@#%&=,/'\";:~`\$\^\*\(\)\+\[\]\.\{\}\|\?\<\>\\]+|[^\s]+)")
/usr/local/lib/python3.13/dist-packages/clint/packages/colorama/ansitowin32.py:43: SyntaxWarning: invalid escape sequence '\['
  ANSI_RE = re.compile('\033\[((?:\d|;)*)([a-zA-Z])')
/usr/local/lib/python3.13/dist-packages/clint/textui/prompt.py:68: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if prompt[-1] is not ' ':
downloading Dataset: tgbl-wiki to /kaggle/working/ML_Term_Project/third_party/TGB/tgb/datasets/tgbl_wiki/tgbl-wiki.zip
============*** EdgeBank: unlimited: tgbl-wiki ***==============
INFO: Create directory /kaggle/working/ML_Term_Project/third_party/TGB/examples/linkproppred/tgbl-wiki/saved_results
100%|██████████| 119/119 [00:17<00:00,  6.69it/s]
INFO: val: Evaluation Setting: >>> ONE-VS-MANY <<< 
	val: mrr:  0.5267
	val: Elapsed Time (s):  17.79

In [5]:
for tag, mode in [("inf", "unlimited"), ("tw", "fixed_time_window")]:
    vals = []
    for s in [1, 2, 3]:
        p = glob.glob(f"{REPO_DIR}/results/edgebank_seeds/seed{s}_EdgeBank_{mode}*.json")
        vals.append(json.load(open(p[0]))["mrr"] if p else None)
    print(tag, vals)
    if None not in vals:
        print(f"   mean {st.mean(vals):.4f}  std {st.stdev(vals):.6f}")

TypeError: list indices must be integers or slices, not str

In [6]:
for f in sorted(glob.glob(f"{REPO_DIR}/results/edgebank_seeds/*.json")):
    print(os.path.basename(f))
    print(open(f).read())
    print()

seed1_EdgeBank_fixed_time_window_tgbl-wiki_results.json
{
    "model": "EdgeBank",
    "memory_mode": "fixed_time_window",
    "data": "tgbl-wiki",
    "run": 1,
    "seed": 1,
    "mrr": 0.5710241198539734,
    "test_time": 18.019097328000043,
    "tot_train_val_time": "NA"
}

seed1_EdgeBank_unlimited_tgbl-wiki_results.json
{
    "model": "EdgeBank",
    "memory_mode": "unlimited",
    "data": "tgbl-wiki",
    "run": 1,
    "seed": 1,
    "mrr": 0.4946681559085846,
    "test_time": 18.150049787,
    "tot_train_val_time": "NA"
}

seed2_EdgeBank_fixed_time_window_tgbl-wiki_results.json
[
    {
        "model": "EdgeBank",
        "memory_mode": "fixed_time_window",
        "data": "tgbl-wiki",
        "run": 1,
        "seed": 1,
        "mrr": 0.5710241198539734,
        "test_time": 18.019097328000043,
        "tot_train_val_time": "NA"
    },
    {
        "model": "EdgeBank",
        "memory_mode": "fixed_time_window",
        "data": "tgbl-wiki",
        "run": 1,
        "seed": 2

In [7]:
def entries(path):
    d = json.load(open(path))
    items = d if isinstance(d, list) else [d]
    return [(it.get("seed"), it.get("run"), it.get("mrr")) for it in items]

for f in sorted(glob.glob(f"{REPO_DIR}/results/edgebank_seeds/*.json")):
    print(os.path.basename(f), entries(f))

seed1_EdgeBank_fixed_time_window_tgbl-wiki_results.json [(1, 1, 0.5710241198539734)]
seed1_EdgeBank_unlimited_tgbl-wiki_results.json [(1, 1, 0.4946681559085846)]
seed2_EdgeBank_fixed_time_window_tgbl-wiki_results.json [(1, 1, 0.5710241198539734), (2, 1, 0.5710241198539734)]
seed2_EdgeBank_unlimited_tgbl-wiki_results.json [(1, 1, 0.4946681559085846), (2, 1, 0.4946681559085846)]
seed3_EdgeBank_fixed_time_window_tgbl-wiki_results.json [(1, 1, 0.5710241198539734), (2, 1, 0.5710241198539734), (3, 1, 0.5710241198539734)]
seed3_EdgeBank_unlimited_tgbl-wiki_results.json [(1, 1, 0.4946681559085846), (2, 1, 0.4946681559085846), (3, 1, 0.4946681559085846)]


In [8]:
import re, csv

d = f"{REPO_DIR}/results/edgebank_seeds"
rows = []
for tag, mode in [("inf", "unlimited"), ("tw", "fixed_time_window")]:
    full = json.load(open(f"{d}/seed3_EdgeBank_{mode}_tgbl-wiki_results.json"))   # holds seeds 1-3
    for it in full:
        s = it["seed"]
        log = open(f"{REPO_DIR}/logs/edgebank_{tag}_wiki_seed{s}.txt").read()
        val = float(re.findall(r"val:\s*mrr:\s*([0-9.]+)", log)[0])
        rows.append((tag, s, val, it["mrr"], it["test_time"]))
    shutil.copy(f"{d}/seed3_EdgeBank_{mode}_tgbl-wiki_results.json",
                f"{d}/EdgeBank_{mode}_all_seeds.json")

for f in glob.glob(f"{d}/seed*_EdgeBank_*.json"):   # remove the redundant per-seed copies
    os.remove(f)

with open(f"{REPO_DIR}/results/edgebank_wiki.csv", "w") as f:
    f.write("variant,seed,val_mrr,test_mrr,test_time_s\n")
    for r in rows:
        f.write(",".join(str(x) for x in r) + "\n")

print(open(f"{REPO_DIR}/results/edgebank_wiki.csv").read())
print(sorted(os.listdir(d)))

variant,seed,val_mrr,test_mrr,test_time_s
inf,1,0.5267,0.4946681559085846,18.150049787
inf,2,0.5267,0.4946681559085846,17.839812313999914
inf,3,0.5267,0.4946681559085846,17.712091567000016
tw,1,0.5997,0.5710241198539734,18.019097328000043
tw,2,0.5997,0.5710241198539734,17.87345643900005
tw,3,0.5997,0.5710241198539734,17.960686240999962

['EdgeBank_fixed_time_window_all_seeds.json', 'EdgeBank_unlimited_all_seeds.json']


In [9]:
save_and_push("EdgeBank seeds 1-3 on tgbl-wiki: identical results (deterministic)")


A  logs/edgebank_inf_wiki_seed1.txt
A  logs/edgebank_inf_wiki_seed2.txt
A  logs/edgebank_inf_wiki_seed3.txt
A  logs/edgebank_tw_wiki_seed1.txt
A  logs/edgebank_tw_wiki_seed2.txt
A  logs/edgebank_tw_wiki_seed3.txt
A  results/edgebank_seeds/EdgeBank_fixed_time_window_all_seeds.json
A  results/edgebank_seeds/EdgeBank_unlimited_all_seeds.json
A  results/edgebank_wiki.csv

[main c53cd97] EdgeBank seeds 1-3 on tgbl-wiki: identical results (deterministic)
 9 files changed, 145 insertions(+)
 create mode 100644 logs/edgebank_inf_wiki_seed1.txt
 create mode 100644 logs/edgebank_inf_wiki_seed2.txt
 create mode 100644 logs/edgebank_inf_wiki_seed3.txt
 create mode 100644 logs/edgebank_tw_wiki_seed1.txt
 create mode 100644 logs/edgebank_tw_wiki_seed2.txt
 create mode 100644 logs/edgebank_tw_wiki_seed3.txt
 create mode 100644 results/edgebank_seeds/EdgeBank_fixed_time_window_all_seeds.json
 create mode 100644 results/edgebank_seeds/EdgeBank_unlimited_all_seeds.json
 create mode 100644 results/edgeb

In [10]:
import csv, glob, os, statistics as st

PUB = {  # TGB paper, Table 2a, tgbl-wiki: (val_mean, val_std, test_mean, test_std)
    "EdgeBank_inf": (0.527, None, 0.495, None),
    "EdgeBank_tw":  (0.600, None, 0.571, None),
    "TGN":          (0.435, 0.069, 0.396, 0.060),
    "DyRep":        (0.072, 0.009, 0.050, 0.017),
}

def read_rows(path):
    with open(f"{REPO_DIR}/{path}") as f:
        return list(csv.DictReader(f))

def stats(vals):
    return st.mean(vals), st.stdev(vals)

mine = {}

# TGN and DyRep
for name, p in [("TGN", "results/tgn_wiki.csv"), ("DyRep", "results/dyrep_wiki.csv")]:
    rows = read_rows(p)
    mv, sv = stats([float(r["best_val_mrr"]) for r in rows])
    mt, stt = stats([float(r["test_mrr"]) for r in rows])
    mine[name] = (mv, sv, mt, stt, len(rows))

# EdgeBank (3 seeds)
eb = read_rows("results/edgebank_wiki.csv")
for tag, key in [("inf", "EdgeBank_inf"), ("tw", "EdgeBank_tw")]:
    r = [x for x in eb if x["variant"] == tag]
    mv, sv = stats([float(x["val_mrr"]) for x in r])
    mt, stt = stats([float(x["test_mrr"]) for x in r])
    mine[key] = (mv, sv, mt, stt, len(r))

fmt = lambda a, s: f"{a:.4f} ± {s:.4f}"
lines = ["| Model | Published val | Published test | My val (best) | My test | n | Test gap / published std |",
         "|---|---|---|---|---|---|---|"]
for m in ["EdgeBank_inf", "EdgeBank_tw", "TGN", "DyRep"]:
    pv, pvs, pt, pts = PUB[m]
    mv, sv, mt, stt, n = mine[m]
    pub_v = f"{pv:.3f}" if pvs is None else f"{pv:.3f} ± {pvs:.3f}"
    pub_t = f"{pt:.3f}" if pts is None else f"{pt:.3f} ± {pts:.3f}"
    gap = f"{(mt - pt) / pts:+.2f}" if pts else "n/a (deterministic)"
    lines.append(f"| {m} | {pub_v} | {pub_t} | {fmt(mv, sv)} | {fmt(mt, stt)} | {n} | {gap} |")
lines.append("| GraphMixer | 0.113 ± 0.003 | 0.118 ± 0.002 | not reproduced | not reproduced | 0 | n/a |")

table = "\n".join(lines)
table += ("\n\nPublished values: Huang et al., NeurIPS 2023 D&B, Table 2a (tgbl-wiki, 5 runs). "
          "Mine: seeds 1-3, TGB default settings, official TGB evaluator and negatives. "
          "EdgeBank is deterministic: all three seeds gave identical results.\n")
print(table)
open(f"{REPO_DIR}/notes/baseline_comparison_table.md", "w").write(table)

| Model | Published val | Published test | My val (best) | My test | n | Test gap / published std |
|---|---|---|---|---|---|---|
| EdgeBank_inf | 0.527 | 0.495 | 0.5267 ± 0.0000 | 0.4947 ± 0.0000 | 3 | n/a (deterministic) |
| EdgeBank_tw | 0.600 | 0.571 | 0.5997 ± 0.0000 | 0.5710 ± 0.0000 | 3 | n/a (deterministic) |
| TGN | 0.435 ± 0.069 | 0.396 ± 0.060 | 0.3852 ± 0.0086 | 0.3328 ± 0.0124 | 3 | -1.05 |
| DyRep | 0.072 ± 0.009 | 0.050 ± 0.017 | 0.0508 ± 0.0199 | 0.0399 ± 0.0143 | 3 | -0.60 |
| GraphMixer | 0.113 ± 0.003 | 0.118 ± 0.002 | not reproduced | not reproduced | 0 | n/a |

Published values: Huang et al., NeurIPS 2023 D&B, Table 2a (tgbl-wiki, 5 runs). Mine: seeds 1-3, TGB default settings, official TGB evaluator and negatives. EdgeBank is deterministic: all three seeds gave identical results.



813

In [11]:
save_and_push("Update baseline comparison table: EdgeBank n=3 (deterministic)")


M  notes/baseline_comparison_table.md

[main 5946d94] Update baseline comparison table: EdgeBank n=3 (deterministic)
 1 file changed, 4 insertions(+), 2 deletions(-)

Current branch main is up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/ML_Term_Project.git
   c53cd97..5946d94  main -> main



In [16]:
import subprocess, time
procs = {}
for s, gpu in [(1, 0), (2, 1)]:
    log = f"/kaggle/working/gm_test_seed{s}.txt"
    procs[s] = subprocess.Popen(
        f"cd {BASE} && python train_tgb_lpp_original.py --dataset_name tgbl-wiki --model_name GraphMixer "
        f"--num_epochs 1 --num_runs 1 --seed {s} --gpu {gpu} > {log} 2>&1", shell=True)
    time.sleep(60)   # stagger the starts, in case both write the same data files

In [18]:
!pkill -f train_tgb_lpp_original.py

In [19]:
import subprocess, base64, os, glob, shutil, time, re, json
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/Rosierobber/Temporal-Graph-ML-Project.git"
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/Temporal-Graph-ML-Project"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()

def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", "-c", f"http.extraheader={hdr}", *args],
                       cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    return r.returncode

def save_and_push(msg, paths):
    git("add", *paths)
    git("commit", "-m", msg)
    git("pull", "--rebase", "origin", "main")
    git("push", "origin", "main")

if not os.path.exists(REPO_DIR):
    git("clone", REPO_URL, cwd=WORK)
git("config", "user.name", "Rabia Habib")
git("config", "user.email", "rabihabib0411@gmail.com")
git("pull", "--rebase", "origin", "main")
os.makedirs(f"{REPO_DIR}/results/graphmixer", exist_ok=True)
os.makedirs(f"{REPO_DIR}/logs", exist_ok=True)



error: cannot pull with rebase: You have unstaged changes.
error: Please commit or stash them.



In [20]:
git("status", "--short")

 M .gitignore



0

In [22]:
git("log", "--format=%h  %an  %s", "-6")
print(sorted(os.listdir(f"{REPO_DIR}/results")))
print(open(f"{REPO_DIR}/notes/baseline_comparison_table.md").read()[:600])

a3225ba  Rabia Habib  Ignore cloned TGB_Baselines (260 MB)
5946d94  Rabia Habib  Update baseline comparison table: EdgeBank n=3 (deterministic)
c53cd97  Rabia Habib  EdgeBank seeds 1-3 on tgbl-wiki: identical results (deterministic)
1424134  Rabia Habib  Archive Kaggle session notebook
f38a835  Rabia Habib  Add baseline comparison table (tgbl-wiki, TGB paper Table 2a)
c534ac1  RabiaHabib  file until v3,v4,v5

['.gitkeep', 'EdgeBank_fixed_time_window_tgbl-wiki_results.json', 'EdgeBank_unlimited_tgbl-wiki_results.json', 'dyrep_wiki.csv', 'edgebank_seeds', 'edgebank_wiki.csv', 'graphmixer', 'seed1_DyRep_tgbl-wiki_results.json', 'seed1_TGN_tgbl-wiki_results.json', 'seed2_DyRep_tgbl-wiki_results.json', 'seed2_TGN_tgbl-wiki_results.json', 'seed3_DyRep_tgbl-wiki_results.json', 'seed3_TGN_tgbl-wiki_results.json', 'tgn_wiki.csv']
| Model | Published val | Published test | My val (best) | My test | n | Test gap / published std |
|---|---|---|---|---|---|---|
| EdgeBank_inf | 0.527 | 0.495 | 0.52

In [23]:
import subprocess, base64, os, glob, shutil, time, re, json
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/Rosierobber/Temporal-Graph-ML-Project.git"
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/Temporal-Graph-ML-Project"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()

def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", "-c", f"http.extraheader={hdr}", *args],
                       cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    return r.returncode

def save_and_push(msg, paths):
    git("add", *paths)
    git("commit", "-m", msg)
    git("pull", "--rebase", "origin", "main")
    git("push", "origin", "main")

if not os.path.exists(REPO_DIR):
    git("clone", REPO_URL, cwd=WORK)
git("config", "user.name", "Rabia Habib")
git("config", "user.email", "rabihabib0411@gmail.com")
git("pull", "--rebase", "origin", "main")
os.makedirs(f"{REPO_DIR}/results/graphmixer", exist_ok=True)
os.makedirs(f"{REPO_DIR}/logs", exist_ok=True)



Already up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch            main       -> FETCH_HEAD



In [25]:
src_note = f"{REPO_DIR}/logs/graphmixer_source.txt"
if not os.path.exists(src_note):
    out = subprocess.run(["git", "ls-remote", "https://github.com/fpour/TGB_Baselines.git", "HEAD"],
                         capture_output=True, text=True).stdout.split()
    head = out[0] if out else "unknown"
    open(src_note, "w").write(
        "fpour/TGB_Baselines was cloned earlier in this session; its exact commit was not recorded.\n"
        f"Remote HEAD when this note was written: {head}\n"
        "epochs per seed: 1\n")

gi = f"{REPO_DIR}/.gitignore"
lines = open(gi).read().splitlines() if os.path.exists(gi) else []
if "third_party/TGB_Baselines/" not in lines:
    with open(gi, "a") as f:
        f.write("third_party/TGB_Baselines/\n")

print(open(src_note).read())
git("status", "--short")

fpour/TGB_Baselines was cloned earlier in this session; its exact commit was not recorded.
Remote HEAD when this note was written: d2022fae4a3311185f611bab733e267938251f42
epochs per seed: 1

?? logs/graphmixer_source.txt



0

In [26]:
BASE = f"{REPO_DIR}/third_party/TGB_Baselines"
print("train script present:", os.path.exists(f"{BASE}/train_tgb_lpp_original.py"))
stale = glob.glob(f"{BASE}/saved_results/GraphMixer/**/*.json", recursive=True)
print("stale GraphMixer result files:", stale)
for f in stale:
    os.remove(f)
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

train script present: True
stale GraphMixer result files: []
index, utilization.gpu [%], memory.used [MiB]
0, 0 %, 3 MiB
1, 0 %, 3 MiB


In [ ]:
import threading

EPOCHS, SEEDS = 1, [1, 2, 3]
BLOCKING = True    # True for Save & Run All, False if you run it interactively
RES = f"{REPO_DIR}/results/graphmixer"
BASE = f"{REPO_DIR}/third_party/TGB_Baselines"

def result_path(s): return f"{RES}/seed{s}_GraphMixer_tgbl-wiki.json"
def log_path(s):    return f"{REPO_DIR}/logs/graphmixer_wiki_seed{s}.txt"

def launch(s, gpu):
    cmd = (f"cd {BASE} && python train_tgb_lpp_original.py --dataset_name tgbl-wiki "
           f"--model_name GraphMixer --num_epochs {EPOCHS} --num_runs 1 --seed {s} --gpu {gpu} "
           f"> {log_path(s)} 2>&1")
    return subprocess.Popen(cmd, shell=True)

def finish(s):
    if "Test: mrr" not in open(log_path(s), errors="ignore").read():
        print(f"seed {s}: no final test MRR in the log, not saved. Check {log_path(s)}")
        return
    src = glob.glob(f"{BASE}/saved_results/GraphMixer/tgbl-wiki*/GraphMixer_tgbl-wiki_seed_{s}_run_0*.json")
    if not src:
        print(f"seed {s}: finished but no result JSON found; the log is still saved")
    else:
        shutil.copy(src[0], result_path(s))
    save_and_push(f"GraphMixer tgbl-wiki seed {s} ({EPOCHS} epoch)",
                  [f"logs/graphmixer_wiki_seed{s}.txt", "logs/graphmixer_source.txt", "results/graphmixer", ".gitignore"])

def run_all():
    pending = [s for s in SEEDS if not os.path.exists(result_path(s))]
    print("seeds to run:", pending)
    running, free_gpus = {}, [0, 1]
    while pending or running:
        while pending and free_gpus:
            s, g = pending.pop(0), free_gpus.pop(0)
            running[s] = (launch(s, g), g)
            print(f"started seed {s} on GPU {g} at {time.strftime('%H:%M:%S')}")
            time.sleep(180)          # stagger starts so the two runs don't write data files together
        for s, (p, g) in list(running.items()):
            if p.poll() is not None:
                print(f"seed {s} ended (exit code {p.returncode}) at {time.strftime('%H:%M:%S')}")
                finish(s)
                free_gpus.append(g)
                del running[s]
        time.sleep(30)
    print("all done")

if BLOCKING:
    run_all()
else:
    threading.Thread(target=run_all).start()

seeds to run: [1, 2, 3]
started seed 1 on GPU 0 at 09:57:30
started seed 2 on GPU 1 at 10:00:30
seed 1 ended (exit code 0) at 12:02:00

[main cd06a65] GraphMixer tgbl-wiki seed 1 (1 epoch)
 3 files changed, 88 insertions(+)
 create mode 100644 logs/graphmixer_source.txt
 create mode 100644 logs/graphmixer_wiki_seed1.txt
 create mode 100644 results/graphmixer/seed1_GraphMixer_tgbl-wiki.json

Current branch main is up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/ML_Term_Project.git
   a3225ba..cd06a65  main -> main

started seed 3 on GPU 0 at 12:02:31
seed 2 ended (exit code 0) at 12:06:01

[main af65e79] GraphMixer tgbl-wiki seed 2 (1 epoch)
 2 files changed, 85 insertions(+)
 create mode 100644 logs/graphmixer_wiki_seed2.txt
 create mode 100644 results/graphmixer/seed2_GraphMixer_tgbl-wiki.json

Current branch main is up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch

In [29]:
BASE = f"{REPO_DIR}/third_party/TGB_Baselines"
s = 3
res  = f"{REPO_DIR}/results/graphmixer/seed{s}_GraphMixer_tgbl-wiki.json"
logp = f"{REPO_DIR}/logs/graphmixer_wiki_seed{s}.txt"

if os.path.exists(res):
    print("seed 3 result already in the repo folder")
else:
    src = glob.glob(f"{BASE}/saved_results/GraphMixer/tgbl-wiki*/GraphMixer_tgbl-wiki_seed_{s}_run_0*.json")
    if src:
        os.makedirs(os.path.dirname(res), exist_ok=True)
        shutil.copy(src[0], res)
        print("copied", os.path.basename(src[0]))
    else:
        print("no result JSON found; the log alone will be pushed")

save_and_push("GraphMixer tgbl-wiki seed 3 (1 epoch)",
              [f"logs/graphmixer_wiki_seed{s}.txt", "results/graphmixer"])

seed 3 result already in the repo folder

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean

Already up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch            main       -> FETCH_HEAD

Everything up-to-date



In [30]:
git("log", "--format=%h  %an  %s", "-6")
print(sorted(os.listdir(f"{REPO_DIR}/results/graphmixer")))
print(sorted(f for f in os.listdir(f"{REPO_DIR}/logs") if "graphmixer" in f))

5c38552  Rabia Habib  GraphMixer tgbl-wiki seed 3 (1 epoch)
af65e79  Rabia Habib  GraphMixer tgbl-wiki seed 2 (1 epoch)
cd06a65  Rabia Habib  GraphMixer tgbl-wiki seed 1 (1 epoch)
a3225ba  Rabia Habib  Ignore cloned TGB_Baselines (260 MB)
5946d94  Rabia Habib  Update baseline comparison table: EdgeBank n=3 (deterministic)
c53cd97  Rabia Habib  EdgeBank seeds 1-3 on tgbl-wiki: identical results (deterministic)

['seed1_GraphMixer_tgbl-wiki.json', 'seed2_GraphMixer_tgbl-wiki.json', 'seed3_GraphMixer_tgbl-wiki.json']
['graphmixer_source.txt', 'graphmixer_wiki_seed1.txt', 'graphmixer_wiki_seed2.txt', 'graphmixer_wiki_seed3.txt']


In [31]:
import csv, re, statistics as st
rows = []
for s in [1, 2, 3]:
    t = open(f"{REPO_DIR}/logs/graphmixer_wiki_seed{s}.txt", errors="ignore").read()
    val = re.findall(r"Validation: mrr:\s*([0-9.]+)", t)
    test = re.findall(r"Test: mrr:\s*([0-9.]+)", t)
    tv = re.findall(r"Total train & validation elapsed time \(s\):\s*([0-9.]+)", t)
    tt = re.findall(r"Test elapsed time \(s\):\s*([0-9.]+)", t)
    if val and test:
        rows.append((s, 1, float(val[0]), float(test[0]),
                     float(tv[0]) if tv else "", float(tt[0]) if tt else ""))
    else:
        print(f"seed {s}: could not read val/test MRR from the log")

with open(f"{REPO_DIR}/results/graphmixer_wiki.csv", "w") as f:
    f.write("seed,epochs_run,val_mrr,test_mrr,train_val_time_s,test_time_s\n")
    for r in rows:
        f.write(",".join(str(x) for x in r) + "\n")
print(open(f"{REPO_DIR}/results/graphmixer_wiki.csv").read())
if len(rows) >= 2:
    tests = [r[3] for r in rows]; vals = [r[2] for r in rows]
    print(f"val  MRR mean ± std (n={len(rows)}): {st.mean(vals):.4f} ± {st.stdev(vals):.4f}   published: 0.113 ± 0.003")
    print(f"test MRR mean ± std (n={len(rows)}): {st.mean(tests):.4f} ± {st.stdev(tests):.4f}   published: 0.118 ± 0.002")
save_and_push("GraphMixer tgbl-wiki summary CSV", ["results/graphmixer_wiki.csv"])

seed,epochs_run,val_mrr,test_mrr,train_val_time_s,test_time_s
1,1,0.1896,0.1892,3706.832543,3733.7847
2,1,0.002,0.002,3749.883488,3744.5964
3,1,0.1369,0.139,3398.630256,3283.7495

val  MRR mean ± std (n=3): 0.1095 ± 0.0968   published: 0.113 ± 0.003
test MRR mean ± std (n=3): 0.1101 ± 0.0969   published: 0.118 ± 0.002

[main 823adc2] GraphMixer tgbl-wiki summary CSV
 1 file changed, 4 insertions(+)
 create mode 100644 results/graphmixer_wiki.csv

Current branch main is up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/ML_Term_Project.git
   5c38552..823adc2  main -> main



In [32]:
import re
for s in [1, 2, 3]:
    t = open(f"{REPO_DIR}/logs/graphmixer_wiki_seed{s}.txt", errors="ignore").read().replace("\r", "\n")
    print(f"--- seed {s} ---")
    for line in t.splitlines():
        if any(k in line for k in ["train loss", "train average_precision", "train roc_auc", "Validation: mrr", "Test: mrr", "Error"]):
            print("  ", line.strip()[:150])

--- seed 1 ---
   Epoch: 1, train for the 1-th batch, train loss: 0.7130148410797119:   0%|                       | 0/552 [00:00<?, ?it/s]
   Epoch: 1, train for the 1-th batch, train loss: 0.7130148410797119:   0%|               | 1/552 [00:00<03:30,  2.62it/s]
   Epoch: 1, train for the 2-th batch, train loss: 0.7107955813407898:   0%|               | 1/552 [00:00<03:30,  2.62it/s]
   Epoch: 1, train for the 3-th batch, train loss: 0.7078767418861389:   0%|               | 1/552 [00:00<03:30,  2.62it/s]
   Epoch: 1, train for the 3-th batch, train loss: 0.7078767418861389:   1%|               | 3/552 [00:00<01:28,  6.20it/s]
   Epoch: 1, train for the 4-th batch, train loss: 0.706338107585907:   1%|                | 3/552 [00:00<01:28,  6.20it/s]
   Epoch: 1, train for the 5-th batch, train loss: 0.7058814764022827:   1%|               | 3/552 [00:00<01:28,  6.20it/s]
   Epoch: 1, train for the 5-th batch, train loss: 0.7058814764022827:   1%|▏              | 5/552 [00:00<01:01,  8.8

In [38]:
open(f"{REPO_DIR}/notes/graphmixer_seed2_failed_run.md", "w").write("""# GraphMixer tgbl-wiki, seed 2: run that did not train

- Seed 2 (GPU 1): train ROC AUC 0.5001, train AP 0.5001, val/test MRR 0.0020.
- Check run, seed 2 on GPU 0 (first 200 batches only, stopped, not saved as a result):
  training losses at batches 1, 50, 100, 200 identical to the original run
  (0.7142066955566406, 0.7137110829353333, 0.7132154703140259, 0.7122503519058228).
- So the failure repeats on a different GPU; it is tied to this seed, not to GPU 1.
- Cause not identified.
- Rule fixed before running further seeds: a run counts as failed if train ROC AUC < 0.55
  after the epoch. All runs are reported; headline statistics use non-failed runs.
""")
save_and_push("Add note on GraphMixer seed 2 failed run", ["notes/graphmixer_seed2_failed_run.md"])


[main b66989a] Add note on GraphMixer seed 2 failed run
 1 file changed, 10 insertions(+)
 create mode 100644 notes/graphmixer_seed2_failed_run.md

Current branch main is up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/ML_Term_Project.git
   5e88e06..b66989a  main -> main



In [36]:
EPOCHS, SEEDS = 1, [4, 5]          # changed: seeds 4 and 5
BLOCKING = True                    # True for Save & Run All or if you keep the tab open
RES = f"{REPO_DIR}/results/graphmixer"

def result_path(s): return f"{RES}/seed{s}_GraphMixer_tgbl-wiki.json"
def log_path(s):    return f"{REPO_DIR}/logs/graphmixer_wiki_seed{s}.txt"

def launch(s, gpu):
    cmd = (f"cd {BASE} && python train_tgb_lpp_original.py --dataset_name tgbl-wiki "
           f"--model_name GraphMixer --num_epochs {EPOCHS} --num_runs 1 --seed {s} --gpu {gpu} "
           f"> {log_path(s)} 2>&1")
    return subprocess.Popen(cmd, shell=True)

def finish(s):
    if "Test: mrr" not in open(log_path(s), errors="ignore").read():
        print(f"seed {s}: no final test MRR in the log, not saved. Check {log_path(s)}")
        return
    src = glob.glob(f"{BASE}/saved_results/GraphMixer/tgbl-wiki*/GraphMixer_tgbl-wiki_seed_{s}_run_0*.json")
    if not src:
        print(f"seed {s}: finished but no result JSON found; the log is still saved")
    else:
        shutil.copy(src[0], result_path(s))
    save_and_push(f"GraphMixer tgbl-wiki seed {s} ({EPOCHS} epoch)",
                  [f"logs/graphmixer_wiki_seed{s}.txt", "results/graphmixer"])

def run_all():
    pending = [s for s in SEEDS if not os.path.exists(result_path(s))]
    print("seeds to run:", pending)
    running, free_gpus = {}, [0, 1]
    while pending or running:
        while pending and free_gpus:
            s, g = pending.pop(0), free_gpus.pop(0)
            running[s] = (launch(s, g), g)
            print(f"started seed {s} on GPU {g} at {time.strftime('%H:%M:%S')}")
            time.sleep(180)
        for s, (p, g) in list(running.items()):
            if p.poll() is not None:
                print(f"seed {s} ended (exit code {p.returncode}) at {time.strftime('%H:%M:%S')}")
                finish(s)
                free_gpus.append(g)
                del running[s]
        time.sleep(30)
    print("all done")

if BLOCKING:
    run_all()
else:
    threading.Thread(target=run_all).start()

seeds to run: [4, 5]
started seed 4 on GPU 0 at 14:25:22
started seed 5 on GPU 1 at 14:28:22
seed 4 ended (exit code 0) at 16:29:52

[main 7b9a966] GraphMixer tgbl-wiki seed 4 (1 epoch)
 2 files changed, 85 insertions(+)
 create mode 100644 logs/graphmixer_wiki_seed4.txt
 create mode 100644 results/graphmixer/seed4_GraphMixer_tgbl-wiki.json

Current branch main is up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/ML_Term_Project.git
   823adc2..7b9a966  main -> main

seed 5 ended (exit code 0) at 16:32:23

[main 793c328] GraphMixer tgbl-wiki seed 5 (1 epoch)
 2 files changed, 85 insertions(+)
 create mode 100644 logs/graphmixer_wiki_seed5.txt
 create mode 100644 results/graphmixer/seed5_GraphMixer_tgbl-wiki.json

Current branch main is up to date.
From https://github.com/Rosierobber/ML_Term_Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/ML_Term_Projec

In [2]:
import csv, statistics as st

PUB = {
    "EdgeBank_inf": (0.527, None, 0.495, None),
    "EdgeBank_tw":  (0.600, None, 0.571, None),
    "TGN":          (0.435, 0.069, 0.396, 0.060),
    "DyRep":        (0.072, 0.009, 0.050, 0.017),
    "GraphMixer":   (0.113, 0.003, 0.118, 0.002),
}
def read_rows(p):
    with open(f"{REPO_DIR}/{p}") as f: return list(csv.DictReader(f))
def ms(v): return st.mean(v), (st.stdev(v) if len(v) > 1 else 0.0)

mine = {}
for name, p in [("TGN", "results/tgn_wiki.csv"), ("DyRep", "results/dyrep_wiki.csv")]:
    r = read_rows(p)
    mine[name] = (*ms([float(x["best_val_mrr"]) for x in r]), *ms([float(x["test_mrr"]) for x in r]), len(r))
eb = read_rows("results/edgebank_wiki.csv")
for tag, key in [("inf", "EdgeBank_inf"), ("tw", "EdgeBank_tw")]:
    r = [x for x in eb if x["variant"] == tag]
    mine[key] = (*ms([float(x["val_mrr"]) for x in r]), *ms([float(x["test_mrr"]) for x in r]), len(r))
gm = read_rows("results/graphmixer_wiki.csv")
ok = [x for x in gm if x["failed"] == "False"]
mine["GraphMixer"] = (*ms([float(x["val_mrr"]) for x in ok]), *ms([float(x["test_mrr"]) for x in ok]), len(ok))
gm_all_t = ms([float(x["test_mrr"]) for x in gm])

fmt = lambda a, s: f"{a:.4f} ± {s:.4f}"
lines = ["| Model | Published val | Published test | My val | My test | n | Test gap / published std |",
         "|---|---|---|---|---|---|---|"]
for m in ["EdgeBank_inf", "EdgeBank_tw", "TGN", "DyRep", "GraphMixer"]:
    pv, pvs, pt, pts = PUB[m]
    mv, sv, mt, stt, n = mine[m]
    pub_v = f"{pv:.3f}" if pvs is None else f"{pv:.3f} ± {pvs:.3f}"
    pub_t = f"{pt:.3f}" if pts is None else f"{pt:.3f} ± {pts:.3f}"
    gap = f"{(mt - pt) / pts:+.2f}" if pts else "n/a (deterministic)"
    label = m + (" *" if m == "GraphMixer" else "")
    lines.append(f"| {label} | {pub_v} | {pub_t} | {fmt(mv, sv)} | {fmt(mt, stt)} | {n} | {gap} |")

table = "\n".join(lines) + f"""

Published values: Huang et al., NeurIPS 2023 D&B, Table 2a (tgbl-wiki, 5 runs).
Mine: TGB default settings and official evaluator/negatives. EdgeBank is deterministic (3 seeds, identical results).
TGN and DyRep: seeds 1-3. "My val" for TGN/DyRep is the best validation epoch; for GraphMixer it is the single epoch's validation MRR.
* GraphMixer: fpour/TGB_Baselines (DyGLib-based), 1 epoch per seed (limited budget), seeds 1-5. Seed 2 did not train
  (train ROC AUC 0.50, test MRR 0.002) and is excluded from this row; it is reported in results/graphmixer_wiki.csv.
  All 5 runs: test MRR {gm_all_t[0]:.4f} ± {gm_all_t[1]:.4f}. The published std for GraphMixer (0.002) is much smaller than the spread
  across my runs, so the gap column is not meaningful for this row.
"""
print(table)
open(f"{REPO_DIR}/notes/baseline_comparison_table.md", "w").write(table)
save_and_push("Update comparison table: add GraphMixer (1 epoch, seed 2 failed)", ["notes/baseline_comparison_table.md"])

| Model | Published val | Published test | My val | My test | n | Test gap / published std |
|---|---|---|---|---|---|---|
| EdgeBank_inf | 0.527 | 0.495 | 0.5267 ± 0.0000 | 0.4947 ± 0.0000 | 3 | n/a (deterministic) |
| EdgeBank_tw | 0.600 | 0.571 | 0.5997 ± 0.0000 | 0.5710 ± 0.0000 | 3 | n/a (deterministic) |
| TGN | 0.435 ± 0.069 | 0.396 ± 0.060 | 0.3852 ± 0.0086 | 0.3328 ± 0.0124 | 3 | -1.05 |
| DyRep | 0.072 ± 0.009 | 0.050 ± 0.017 | 0.0508 ± 0.0199 | 0.0399 ± 0.0143 | 3 | -0.60 |
| GraphMixer * | 0.113 ± 0.003 | 0.118 ± 0.002 | 0.1278 ± 0.0513 | 0.1255 ± 0.0516 | 4 | +3.75 |

Published values: Huang et al., NeurIPS 2023 D&B, Table 2a (tgbl-wiki, 5 runs).
Mine: TGB default settings and official evaluator/negatives. EdgeBank is deterministic (3 seeds, identical results).
TGN and DyRep: seeds 1-3. "My val" for TGN/DyRep is the best validation epoch; for GraphMixer it is the single epoch's validation MRR.
* GraphMixer: fpour/TGB_Baselines (DyGLib-based), 1 epoch per seed (limited budg

In [4]:
!python {REPO_DIR}/scripts/reproduce_table.py

| Model | Published val | Published test | My val | My test | n | Test gap / published std |
|---|---|---|---|---|---|---|
| EdgeBank_inf | 0.527 | 0.495 | 0.5267 ± 0.0000 | 0.4947 ± 0.0000 | 3 | n/a (deterministic) |
| EdgeBank_tw | 0.600 | 0.571 | 0.5997 ± 0.0000 | 0.5710 ± 0.0000 | 3 | n/a (deterministic) |
| TGN | 0.435 ± 0.069 | 0.396 ± 0.060 | 0.3852 ± 0.0086 | 0.3328 ± 0.0124 | 3 | -1.05 |
| DyRep | 0.072 ± 0.009 | 0.050 ± 0.017 | 0.0508 ± 0.0199 | 0.0399 ± 0.0143 | 3 | -0.60 |
| GraphMixer * | 0.113 ± 0.003 | 0.118 ± 0.002 | 0.0000 ± 0.0000 | 0.0000 ± 0.0000 | 0 | n/a (deterministic) |

Published values: Huang et al., NeurIPS 2023 D&B, Table 2a (tgbl-wiki, 5 runs).
Mine: TGB default settings and official evaluator/negatives. EdgeBank is deterministic (3 seeds, identical results).
TGN and DyRep: seeds 1-3. "My val" for TGN/DyRep is the best validation epoch; for GraphMixer it is the single epoch's validation MRR.
* GraphMixer: fpour/TGB_Baselines (DyGLib-based), 1 epoch per seed

In [5]:
!grep -rn "num_epochs\|patience\|num_runs" {BASE}/utils/load_configs.py | head -20

60:    parser.add_argument('--num_epochs', type=int,
66:    parser.add_argument('--patience', type=int, default=10,
67:                        help='patience for early stopping')  # original value = 20
72:    parser.add_argument('--num_runs', type=int,
326:    parser.add_argument('--num_epochs', type=int,
332:    parser.add_argument('--patience', type=int, default=20,
333:                        help='patience for early stopping')
338:    parser.add_argument('--num_runs', type=int,


In [6]:
!sed -n 55,75p {BASE}/utils/load_configs.py
!echo ------
!sed -n 320,342p {BASE}/utils/load_configs.py
!echo ------
!grep -n "^def \|get_link_prediction\|get_.*_args\|load_configs\|from utils" {BASE}/utils/load_configs.py | head -20
!grep -n "load_configs\|get_.*args\|num_epochs\|patience" {BASE}/train_tgb_lpp_original.py | head -20

    parser.add_argument('--learning_rate', type=float,
                        default=0.0001, help='learning rate')
    parser.add_argument('--dropout', type=float,
                        default=0.1, help='dropout rate')
    # original value = 100, --> TGB:50
    parser.add_argument('--num_epochs', type=int,
                        default=25, help='number of epochs')
    parser.add_argument('--optimizer', type=str, default='Adam',
                        choices=['SGD', 'Adam', 'RMSprop'], help='name of optimizer')
    parser.add_argument('--weight_decay', type=float,
                        default=0.0, help='weight decay')
    parser.add_argument('--patience', type=int, default=10,
                        help='patience for early stopping')  # original value = 20
    parser.add_argument('--val_ratio', type=float,
                        default=0.15, help='ratio of validation set')
    parser.add_argument('--test_ratio', type=float,
                        default=0.15, help='rat

In [7]:
!pip install -q py-tgb torch_geometric

need = ["results", "scripts", "notes", "logs", "figures", "third_party/TGB",
        "scripts/reproduce_table.py", "results/tgn_wiki.csv", "results/dyrep_wiki.csv",
        "results/edgebank_wiki.csv", "results/graphmixer_wiki.csv", "results/graphmixer", "README.md"]
for p in need:
    print(f"{p:32s}", "OK" if os.path.exists(f"{REPO_DIR}/{p}") else "MISSING")
print("root:", sorted(os.listdir(REPO_DIR)))
git("log", "--oneline", "-8"); git("status", "--short")

results                          OK
scripts                          OK
notes                            OK
logs                             OK
figures                          OK
third_party/TGB                  OK
scripts/reproduce_table.py       OK
results/tgn_wiki.csv             OK
results/dyrep_wiki.csv           OK
results/edgebank_wiki.csv        OK
results/graphmixer_wiki.csv      OK
results/graphmixer               OK
README.md                        OK
root: ['.git', '.gitignore', 'ML_TERM_PROJECT_1', 'README.md', 'code', 'configs', 'figures', 'logs', 'notebooks', 'notes', 'reproduce_main_table.sh', 'requirements.txt', 'results', 'scripts', 'third_party']
c4eca10 Update README.md..
66e2e8e code file
4c95287 Create dummyfile.py
43a3ff0 Update figures to journal-quality minimalist IEEE styling
432ba9e Generate and push all 4 publication analysis figures dynamically from results/
96279a0 Add ablation study, alpha sensitivity curve, and figure
7cacda0 RQ2 edge recurrence analysi

0

In [8]:
DRY_RUN = True   # run once like this, then set False and run again

targets = [
    "notes/rq2_recurrence_analysis.md",
    "results/edge_recurrence_tgbl_wiki.json",
    "notes/ablation_sensitivity_study.md",
    "results/alpha_sensitivity_tgbl_wiki.csv",
    "figures/alpha_sensitivity.png",
]
targets += [os.path.relpath(p, REPO_DIR) for p in glob.glob(f"{REPO_DIR}/figures/*.png")]
existing = sorted(set(p for p in targets if os.path.exists(f"{REPO_DIR}/{p}")))
print("\n".join(existing) or "nothing to delete")

if not DRY_RUN and existing:
    git("rm", "-f", *existing)
    git("commit", "-m", "Remove outputs built on hardcoded values and invalid alpha sweep (to be regenerated)")
    git("pull", "--rebase", "origin", "main")
    git("push", "origin", "main")

figures/alpha_sensitivity.png
figures/fig1_baseline_per_seed.png
figures/fig2_recurrence_slices.png
figures/fig3_alpha_sensitivity.png
figures/fig4_compute_tradeoff.png
notes/ablation_sensitivity_study.md
notes/rq2_recurrence_analysis.md
results/alpha_sensitivity_tgbl_wiki.csv
results/edge_recurrence_tgbl_wiki.json


In [9]:
os.makedirs(f"{REPO_DIR}/results/archive_graphmixer_1epoch", exist_ok=True)
os.makedirs(f"{REPO_DIR}/logs/archive_graphmixer_1epoch", exist_ok=True)

def mv(src, dst):
    if os.path.exists(f"{REPO_DIR}/{src}"): git("mv", src, dst)
    else: print("skip (missing):", src)

mv("results/graphmixer", "results/archive_graphmixer_1epoch/runs")
mv("results/graphmixer_wiki.csv", "results/archive_graphmixer_1epoch/graphmixer_wiki.csv")
mv("notes/graphmixer_seed2_failed_run.md", "results/archive_graphmixer_1epoch/graphmixer_seed2_failed_run.md")
for p in glob.glob(f"{REPO_DIR}/logs/graphmixer_wiki_seed*.txt"):
    mv(os.path.relpath(p, REPO_DIR), "logs/archive_graphmixer_1epoch/" + os.path.basename(p))

git("commit", "-m", "Archive 1-epoch GraphMixer runs before rerunning with more epochs")
git("pull", "--rebase", "origin", "main")
git("push", "origin", "main")



skip (missing): notes/graphmixer_seed2_failed_run.md





[main 0821005] Archive 1-epoch GraphMixer runs before rerunning with more epochs
 11 files changed, 0 insertions(+), 0 deletions(-)
 rename logs/{ => archive_graphmixer_1epoch}/graphmixer_wiki_seed1.txt (100%)
 rename logs/{ => archive_graphmixer_1epoch}/graphmixer_wiki_seed2.txt (100%)
 rename logs/{ => archive_graphmixer_1epoch}/graphmixer_wiki_seed3.txt (100%)
 rename logs/{ => archive_graphmixer_1epoch}/graphmixer_wiki_seed4.txt (100%)
 rename logs/{ => archive_graphmixer_1epoch}/graphmixer_wiki_seed5.txt (100%)
 rename results/{ => archive_graphmixer_1epoch}/graphmixer_wiki.csv (100%)
 rename results/{graphmixer => archive_graphmixer_1epoch/runs}/seed1_GraphMixer_tgbl-wiki.json (100%)
 rename results/{graphmixer => archive_graphmixer_1epoch/runs}/seed2_GraphMixer_tgbl-wiki.json (100%)
 rename results/{graphmixer => archive_graphmixer_1epoch/runs}/seed3_GraphMixer_tgbl-wiki.json (100%)
 rename results/{graphmixer => archiv

0

In [18]:
import threading, time, re

BASE   = f"{REPO_DIR}/third_party/TGB_Baselines"
EPOCHS = 5
SEEDS  = [1, 2]
RES = f"{REPO_DIR}/results/graphmixer"
os.makedirs(RES, exist_ok=True); os.makedirs(f"{REPO_DIR}/logs", exist_ok=True)

def result_path(s): return f"{RES}/seed{s}_GraphMixer_tgbl-wiki.json"
def log_path(s):    return f"{REPO_DIR}/logs/graphmixer_wiki_seed{s}.txt"

def launch(s, gpu):
    cmd = (f"cd {BASE} && python train_tgb_lpp_original.py --dataset_name tgbl-wiki "
           f"--model_name GraphMixer --num_epochs {EPOCHS} --num_runs 1 --seed {s} --gpu {gpu} "
           f"> {log_path(s)} 2>&1")
    return subprocess.Popen(cmd, shell=True)

def progress(seeds):
    print(f"--- progress at {time.strftime('%H:%M:%S')} ---")
    for s in seeds:
        p = log_path(s)
        if not os.path.exists(p): print(f"seed {s}: no log yet"); continue
        t = open(p, errors="ignore").read().replace("\r", "\n")
        b = re.findall(r"train for the (\d+)-th batch, train loss: ([0-9.]+)", t)
        ep = len(re.findall(r"Epoch:\s*\d+", t))
        print(f"seed {s}: " + (f"batch {b[-1][0]}, loss {b[-1][1]}, epoch lines {ep}" if b else "no batch lines yet"))

def finish(s):
    if "Test: mrr" not in open(log_path(s), errors="ignore").read():
        print(f"seed {s}: no final test MRR, not saved. Check {log_path(s)}"); return
    src = glob.glob(f"{BASE}/saved_results/GraphMixer/tgbl-wiki*/GraphMixer_tgbl-wiki_seed_{s}_run_0*.json")
    if src: shutil.copy(src[0], result_path(s))
    else:   print(f"seed {s}: no result JSON found; log is still saved")
    save_and_push(f"GraphMixer tgbl-wiki seed {s} ({EPOCHS} epochs)",
                  [f"logs/graphmixer_wiki_seed{s}.txt", "results/graphmixer"])

def run_all():
    pending = [s for s in SEEDS if not os.path.exists(result_path(s))]
    print("seeds to run:", pending)
    running, free_gpus, last_report = {}, [0, 1], 0
    while pending or running:
        while pending and free_gpus:
            s, g = pending.pop(0), free_gpus.pop(0)
            running[s] = (launch(s, g), g)
            print(f"started seed {s} on GPU {g} at {time.strftime('%H:%M:%S')}")
            time.sleep(180)
        for s, (p, g) in list(running.items()):
            if p.poll() is not None:
                print(f"seed {s} ended (exit {p.returncode}) at {time.strftime('%H:%M:%S')}")
                finish(s); free_gpus.append(g); del running[s]
        if running and time.time() - last_report > 600:
            progress(list(running)); last_report = time.time()
        time.sleep(30)
    print("all done")

run_all()

seeds to run: [1, 2]
started seed 1 on GPU 0 at 10:58:32
started seed 2 on GPU 1 at 11:01:32
--- progress at 11:04:32 ---
seed 1: batch 552, loss 0.6453964114189148, epoch lines 829
seed 2: batch 552, loss 0.7091186046600342, epoch lines 829
--- progress at 11:14:32 ---
seed 1: batch 552, loss 0.6453964114189148, epoch lines 829
seed 2: batch 552, loss 0.7091186046600342, epoch lines 829
--- progress at 11:24:32 ---
seed 1: batch 552, loss 0.6453964114189148, epoch lines 829
seed 2: batch 552, loss 0.7091186046600342, epoch lines 829
--- progress at 11:34:32 ---
seed 1: batch 552, loss 0.6453964114189148, epoch lines 829
seed 2: batch 552, loss 0.7091186046600342, epoch lines 829
--- progress at 11:44:32 ---
seed 1: batch 552, loss 0.6453964114189148, epoch lines 829
seed 2: batch 552, loss 0.7091186046600342, epoch lines 829
--- progress at 11:54:32 ---
seed 1: batch 552, loss 0.6453964114189148, epoch lines 829
seed 2: batch 552, loss 0.7091186046600342, epoch lines 829
--- progress 

In [19]:
import csv, re, statistics as st

def last(pattern, text):
    m = re.findall(pattern, text); return float(m[-1]) if m else None

rows = []
for s in range(1, 6):
    p = log_path(s)
    if not os.path.exists(p): continue
    t = open(p, errors="ignore").read().replace("\r", "\n")
    val  = last(r"Validation: mrr:\s*([0-9.]+)", t)
    test = last(r"Test: mrr:\s*([0-9.]+)", t)
    auc  = last(r"train roc_auc,\s*([0-9.]+)", t)
    tv   = last(r"Total train & validation elapsed time \(s\):\s*([0-9.]+)", t)
    tt   = last(r"Test elapsed time \(s\):\s*([0-9.]+)", t)
    if val is None or test is None or auc is None:
        print(f"seed {s}: log incomplete, skipped"); continue
    rows.append((s, EPOCHS, auc, auc < 0.55, val, test, tv, tt))

with open(f"{REPO_DIR}/results/graphmixer_wiki.csv", "w") as f:
    f.write("seed,epochs_run,train_roc_auc,failed,val_mrr,test_mrr,train_val_time_s,test_time_s\n")
    for r in rows: f.write(",".join("" if x is None else str(x) for x in r) + "\n")
print(open(f"{REPO_DIR}/results/graphmixer_wiki.csv").read())

open(f"{REPO_DIR}/logs/graphmixer_source.txt", "w").write(
    "fpour/TGB_Baselines commit: d2022fae4a3311185f611bab733e267938251f42\n"
    f"epochs per seed: {EPOCHS}\n"
    "repo defaults: num_epochs=25, patience=10, test_interval_epochs=10\n"
    "published setup (TGB paper) does not state epochs\n")
save_and_push("GraphMixer tgbl-wiki summary CSV (with failed column)",
              ["results/graphmixer_wiki.csv", "logs/graphmixer_source.txt"])

seed,epochs_run,train_roc_auc,failed,val_mrr,test_mrr,train_val_time_s,test_time_s
1,5,0.8559,False,0.1859,0.1892,19560.500511,3742.2356
2,5,0.5,True,0.002,0.002,19599.05181,3724.9255


M  logs/graphmixer_source.txt
A  results/graphmixer_wiki.csv

[main e8c3a1e] GraphMixer tgbl-wiki summary CSV (with failed column)
 2 files changed, 7 insertions(+), 3 deletions(-)
 create mode 100644 results/graphmixer_wiki.csv

Current branch main is up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/Temporal-Graph-ML-Project.git
   d242b0c..e8c3a1e  main -> main



In [3]:
%%writefile /kaggle/working/Temporal-Graph-ML-Project/scripts/reproduce_table.py
import os, csv, statistics as st
REPO = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))

PUB = {  # Huang et al., NeurIPS 2023 D&B, Table 2a (tgbl-wiki, 5 runs): val, val_std, test, test_std
    "EdgeBank_inf": (0.527, None, 0.495, None), "EdgeBank_tw": (0.600, None, 0.571, None),
    "TGN": (0.435, 0.069, 0.396, 0.060), "DyRep": (0.072, 0.009, 0.050, 0.017),
    "GraphMixer": (0.113, 0.003, 0.118, 0.002)}

def rows(p):
    f = os.path.join(REPO, p)
    return list(csv.DictReader(open(f))) if os.path.exists(f) else []
def ms(v): return (st.mean(v), st.stdev(v) if len(v) > 1 else 0.0) if v else (float("nan"),) * 2
def col(r, c): return [float(x[c]) for x in r]

mine, notes = {}, []
for name, p in [("TGN", "results/tgn_wiki.csv"), ("DyRep", "results/dyrep_wiki.csv")]:
    r = rows(p); mine[name] = (*ms(col(r, "best_val_mrr")), *ms(col(r, "test_mrr")), len(r))
eb = rows("results/edgebank_wiki.csv")
for tag, key in [("inf", "EdgeBank_inf"), ("tw", "EdgeBank_tw")]:
    r = [x for x in eb if x["variant"] == tag]
    mine[key] = (*ms(col(r, "val_mrr")), *ms(col(r, "test_mrr")), len(r))

gm = rows("results/graphmixer_wiki.csv")
failed = [x for x in gm if x.get("failed", "").strip().lower() == "true"]
ok = [x for x in gm if x not in failed]
mine["GraphMixer"] = (*ms(col(ok, "val_mrr")), *ms(col(ok, "test_mrr")), len(ok))
allm, alls = ms(col(gm, "test_mrr"))
if gm and "failed" not in gm[0]:
    notes.append("WARNING: graphmixer_wiki.csv has no 'failed' column; all runs counted as OK.")

fmt = lambda a, s: f"{a:.4f} ± {s:.4f}"
out = ["| Model | Published val | Published test | My val | My test | n | Test gap / published std |",
       "|---|---|---|---|---|---|---|"]
for m in PUB:
    pv, pvs, pt, pts = PUB[m]; mv, sv, mt, stt, n = mine[m]
    pubv = f"{pv:.3f}" if pvs is None else f"{pv:.3f} ± {pvs:.3f}"
    pubt = f"{pt:.3f}" if pts is None else f"{pt:.3f} ± {pts:.3f}"
    gap = "n/a (deterministic)" if pts is None else (f"{(mt - pt) / pts:+.2f}" if n else "n/a")
    out.append(f"| {m} | {pubv} | {pubt} | {fmt(mv, sv)} | {fmt(mt, stt)} | {n} | {gap} |")
out.append("")
out.append(f"Published: TGB paper Table 2a, 5 runs. Mine: official TGB negatives/evaluator; "
           f"TGN/DyRep n={mine['TGN'][4]}, EdgeBank deterministic.")
if gm:
    out.append(f"GraphMixer: {len(gm)} runs, {len(failed)} failed (train ROC AUC < 0.55), "
               f"epochs_run={gm[0].get('epochs_run', '?')}. All runs: test MRR {fmt(allm, alls)}.")
out += notes
text = "\n".join(out); print(text)
os.makedirs(os.path.join(REPO, "notes"), exist_ok=True)
open(os.path.join(REPO, "notes", "baseline_comparison_table.md"), "w").write(text + "\n")

Overwriting /kaggle/working/Temporal-Graph-ML-Project/scripts/reproduce_table.py


In [4]:
!python {REPO_DIR}/scripts/reproduce_table.py
save_and_push("Add/refresh reproduce_table.py and baseline table",
              ["scripts/reproduce_table.py", "notes/baseline_comparison_table.md"])

| Model | Published val | Published test | My val | My test | n | Test gap / published std |
|---|---|---|---|---|---|---|
| EdgeBank_inf | 0.527 | 0.495 | 0.5267 ± 0.0000 | 0.4947 ± 0.0000 | 3 | n/a (deterministic) |
| EdgeBank_tw | 0.600 | 0.571 | 0.5997 ± 0.0000 | 0.5710 ± 0.0000 | 3 | n/a (deterministic) |
| TGN | 0.435 ± 0.069 | 0.396 ± 0.060 | 0.3852 ± 0.0086 | 0.3328 ± 0.0124 | 3 | -1.05 |
| DyRep | 0.072 ± 0.009 | 0.050 ± 0.017 | 0.0508 ± 0.0199 | 0.0399 ± 0.0143 | 3 | -0.60 |
| GraphMixer | 0.113 ± 0.003 | 0.118 ± 0.002 | 0.1635 ± 0.0229 | 0.1577 ± 0.0325 | 3 | +19.87 |

Published: TGB paper Table 2a, 5 runs. Mine: official TGB negatives/evaluator; TGN/DyRep n=3, EdgeBank deterministic.
GraphMixer: 4 runs, 1 failed (train ROC AUC < 0.55), epochs_run=5. All runs: test MRR 0.1188 ± 0.0823.

M  notes/baseline_comparison_table.md
M  scripts/reproduce_table.py

[main 1f9af3b] Add/refresh reproduce_table.py and baseline table
 2 files changed, 44 insertions(+), 51 deletions(-)

Curr

In [5]:
import sys, torch, platform, importlib.metadata as md
def v(p):
    try: return md.version(p)
    except Exception: return "n/a"
info = [f"python {sys.version.split()[0]}", f"torch {torch.__version__}",
        f"torch_geometric {v('torch_geometric')}", f"py-tgb {v('py-tgb')}", f"platform {platform.platform()}"]
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
open(f"{REPO_DIR}/logs/environment.txt", "w").write("\n".join(info) + "\nGPUs:\n" + gpu + "\n")
print(open(f"{REPO_DIR}/logs/environment.txt").read())
save_and_push("Record environment and hardware", ["logs/environment.txt"])

python 3.13.16
torch 2.11.0+cu130
torch_geometric 2.8.1
py-tgb 2.3.0
platform Linux-6.18.55+-x86_64-with-glibc2.39
GPUs:
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


A  logs/environment.txt

[main 3207ce5] Record environment and hardware
 1 file changed, 8 insertions(+)
 create mode 100644 logs/environment.txt

Current branch main is up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/Temporal-Graph-ML-Project.git
   1f9af3b..3207ce5  main -> main



In [6]:
%%writefile /kaggle/working/Temporal-Graph-ML-Project/scripts/redgebank.py
import sys, math, time
import numpy as np

def load(name, tgb_path):
    if tgb_path not in sys.path: sys.path.insert(0, tgb_path)
    from tgb.linkproppred.dataset import LinkPropPredDataset
    from tgb.linkproppred.evaluate import Evaluator
    return LinkPropPredDataset(name=name, root="datasets"), Evaluator(name=name)

class Memory:
    """mode: 'inf' (EdgeBank-inf), 'tw' (EdgeBank-tw), 'recency' (recency inside window only), 'full' (R-EdgeBank)"""
    def __init__(self, rho, kappa, mode):
        self.rho, self.kappa, self.mode, self.M = rho, kappa, mode, {}
    def init(self, s, d, t):
        for a, b, c in zip(s.tolist(), d.tolist(), t.tolist()): self.M[(a, b)] = c
        self.W = self.rho * float(t.max() - t.min()); self.t_c = float(t.max())
    def update(self, s, d, t):
        for a, b, c in zip(s.tolist(), d.tolist(), t.tolist()): self.M[(a, b)] = c
        self.t_c = max(self.t_c, float(t.max()))
    def score(self, s, cands, t):
        M, W, lo = self.M, max(self.W, 1e-12), self.t_c - self.W
        out = np.zeros(len(cands), dtype=np.float64)
        for i, u in enumerate(cands.tolist()):
            lam = M.get((s, u))
            if lam is None: continue
            if self.mode == "inf": out[i] = 1.0
            elif lam >= lo: out[i] = 1.0 if self.mode == "tw" else 1.0 + (lam - lo) / W
            elif self.mode == "full": out[i] = math.exp(-(t - lam) / (self.kappa * W))
        return out

def evaluate(ds, ev, split, rho, kappa, mode, max_edges=None, batch=200):
    d = ds.full_data; S, D, T = d["sources"], d["destinations"], d["timestamps"]
    tr = np.where(ds.train_mask)[0]; va = np.where(ds.val_mask)[0]; te = np.where(ds.test_mask)[0]
    if not getattr(ds, "_ns_" + split, False):
        (ds.load_val_ns if split == "val" else ds.load_test_ns)(); setattr(ds, "_ns_" + split, True)
    mem = Memory(rho, kappa, mode); mem.init(S[tr], D[tr], T[tr])
    if split == "test":                                   # replay validation edges into memory
        for b in range(0, len(va), batch):
            i = va[b:b + batch]; mem.update(S[i], D[i], T[i])
    idx = (te if split == "test" else va)
    if max_edges: idx = idx[:max_edges]
    s_, d_, t_ = S[idx], D[idx], T[idx]
    ns = ds.negative_sampler; rr, seen = [], []; t0 = time.time()
    for b in range(0, len(idx), batch):
        sb, db, tb = s_[b:b + batch], d_[b:b + batch], t_[b:b + batch]
        negs = ns.query_batch(sb, db, tb, split_mode=split)
        for j in range(len(sb)):
            cands = np.concatenate([[db[j]], negs[j]])
            sc = mem.score(int(sb[j]), cands, float(tb[j]))
            seen.append((int(sb[j]), int(db[j])) in mem.M)
            rr.append(ev.eval({"y_pred_pos": np.array([sc[0]]), "y_pred_neg": sc[1:],
                               "eval_metric": ["mrr"]})["mrr"])
        mem.update(sb, db, tb)
    rr = np.array(rr)
    return {"mrr": float(rr.mean()), "rr": rr, "seen": np.array(seen), "secs": time.time() - t0, "n": len(rr)}

Writing /kaggle/working/Temporal-Graph-ML-Project/scripts/redgebank.py


In [7]:
import sys, csv, json, importlib, numpy as np
sys.path.insert(0, f"{REPO_DIR}/scripts")
import redgebank; importlib.reload(redgebank)
ds, ev = redgebank.load("tgbl-wiki", f"{REPO_DIR}/third_party/TGB")

RHOS, KAPPAS = [0.05, 0.10, 0.15, 0.25, 0.40], [0.5, 1.0, 2.0, 4.0]
rows = []
for rho in RHOS:
    for mode in ["tw", "recency"]:
        r = redgebank.evaluate(ds, ev, "val", rho, 1.0, mode)
        rows.append((mode, rho, "", r["mrr"])); print(mode, rho, round(r["mrr"], 4))
    for k in KAPPAS:
        r = redgebank.evaluate(ds, ev, "val", rho, k, "full")
        rows.append(("full", rho, k, r["mrr"])); print("full", rho, k, round(r["mrr"], 4))

os.makedirs(f"{REPO_DIR}/results", exist_ok=True)
with open(f"{REPO_DIR}/results/sweep_val_wiki.csv", "w") as f:
    f.write("mode,rho,kappa,val_mrr\n")
    for r in rows: f.write(",".join(map(str, r)) + "\n")

best = lambda m: max([r for r in rows if r[0] == m], key=lambda r: r[3])
bt, br, bf = best("tw"), best("recency"), best("full")
sel = {"tw_rho": bt[1], "rec_rho": br[1], "full_rho": bf[1], "full_kappa": bf[2],
       "val": {"tw": bt[3], "recency": br[3], "full": bf[3]}}
json.dump(sel, open(f"{REPO_DIR}/results/redgebank_selected_wiki.json", "w"), indent=2)
print(sel)
save_and_push("rho/kappa validation sweep on tgbl-wiki",
              ["scripts/redgebank.py", "results/sweep_val_wiki.csv", "results/redgebank_selected_wiki.json"])

/usr/local/lib/python3.13/dist-packages/clint/textui/colored.py:118: SyntaxWarning: invalid escape sequence '\$'
  strip = re.compile("([^-_a-zA-Z0-9!@#%&=,/'\";:~`\$\^\*\(\)\+\[\]\.\{\}\|\?\<\>\\]+|[^\s]+)")
/usr/local/lib/python3.13/dist-packages/clint/packages/colorama/ansitowin32.py:43: SyntaxWarning: invalid escape sequence '\['
  ANSI_RE = re.compile('\033\[((?:\d|;)*)([a-zA-Z])')
/usr/local/lib/python3.13/dist-packages/clint/textui/prompt.py:68: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if prompt[-1] is not ' ':


downloading Dataset: tgbl-wiki to /kaggle/working/Temporal-Graph-ML-Project/third_party/TGB/tgb/datasets/tgbl_wiki/tgbl-wiki.zip
tw 0.05 0.5895
recency 0.05 0.6488
full 0.05 0.5 0.7555
full 0.05 1.0 0.7555
full 0.05 2.0 0.7555
full 0.05 4.0 0.7555
tw 0.1 0.5969
recency 0.1 0.6904
full 0.1 0.5 0.7555
full 0.1 1.0 0.7555
full 0.1 2.0 0.7555
full 0.1 4.0 0.7555
tw 0.15 0.5997
recency 0.15 0.7115
full 0.15 0.5 0.7555
full 0.15 1.0 0.7555
full 0.15 2.0 0.7555
full 0.15 4.0 0.7555
tw 0.25 0.5912
recency 0.25 0.7315
full 0.25 0.5 0.7555
full 0.25 1.0 0.7555
full 0.25 2.0 0.7555
full 0.25 4.0 0.7555
tw 0.4 0.5713
recency 0.4 0.7443
full 0.4 0.5 0.7555
full 0.4 1.0 0.7555
full 0.4 2.0 0.7555
full 0.4 4.0 0.7555
{'tw_rho': 0.15, 'rec_rho': 0.4, 'full_rho': 0.05, 'full_kappa': 0.5, 'val': {'tw': 0.5997411608695984, 'recency': 0.7442865371704102, 'full': 0.755473792552948}}

A  results/redgebank_selected_wiki.json
A  results/sweep_val_wiki.csv
A  scripts/redgebank.py

[main 6d2b7c6] rho/kappa vali

In [8]:
final = f"{REPO_DIR}/results/test_final_wiki.csv"
assert not os.path.exists(final), "Test already evaluated once; do not rerun."
sel = json.load(open(f"{REPO_DIR}/results/redgebank_selected_wiki.json"))
cfg = {"tw_default_rho0.15": ("tw", 0.15, 1.0),
       "tw_tuned_rho":       ("tw", sel["tw_rho"], 1.0),
       "recency_only":       ("recency", sel["rec_rho"], 1.0),
       "full_R-EdgeBank":    ("full", sel["full_rho"], sel["full_kappa"])}
out = []
for name, (mode, rho, k) in cfg.items():
    r = redgebank.evaluate(ds, ev, "test", rho, k, mode)
    np.savez(f"{REPO_DIR}/results/perquery_test_wiki_{name}.npz", rr=r["rr"], seen=r["seen"])
    out.append((name, mode, rho, k, r["mrr"], r["secs"])); print(name, round(r["mrr"], 4), f"{r['secs']:.0f}s")
with open(final, "w") as f:
    f.write("config,mode,rho,kappa,test_mrr,test_secs\n")
    for o in out: f.write(",".join(map(str, o)) + "\n")
save_and_push("Test evaluation of EdgeBank/R-EdgeBank ablation on tgbl-wiki", ["results"])

tw_default_rho0.15 0.571 8s
tw_tuned_rho 0.571 8s
recency_only 0.7069 8s
full_R-EdgeBank 0.7248 8s

A  results/test_final_wiki.csv

[main 3661cb3] Test evaluation of EdgeBank/R-EdgeBank ablation on tgbl-wiki
 1 file changed, 5 insertions(+)
 create mode 100644 results/test_final_wiki.csv

Current branch main is up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/Temporal-Graph-ML-Project.git
   6d2b7c6..3661cb3  main -> main



In [9]:
d = ds.full_data; S, D = d["sources"], d["destinations"]
tr, te = ds.train_mask, ds.test_mask
train_pairs = set(zip(S[tr].tolist(), D[tr].tolist()))
surprise = float(np.mean([p not in train_pairs for p in zip(S[te].tolist(), D[te].tolist())]))
print(f"surprise index, my computation: {surprise:.3f}   (TGB paper: 0.108)")

rows = []
for name in ["tw_default_rho0.15", "full_R-EdgeBank"]:
    z = np.load(f"{REPO_DIR}/results/perquery_test_wiki_{name}.npz"); rr, seen = z["rr"], z["seen"]
    rows.append((name, "recurring", int(seen.sum()), float(rr[seen].mean())))
    rows.append((name, "novel", int((~seen).sum()), float(rr[~seen].mean())))
    print(name, "recurring share", round(seen.mean(), 4), "| MRR rec", round(rr[seen].mean(), 4),
          "| MRR novel", round(rr[~seen].mean(), 4))

# TGN per-query (optional): save_perf_list from TGB's tgn.py test(); see note below
for s in [1, 2, 3]:
    p = f"{REPO_DIR}/results/tgn_perquery_test_seed{s}.npy"
    if os.path.exists(p):
        rr_t = np.load(p); seen = np.load(f"{REPO_DIR}/results/perquery_test_wiki_tw_default_rho0.15.npz")["seen"]
        assert len(rr_t) == len(seen), "per-query length mismatch"
        rows += [(f"TGN_seed{s}", "recurring", int(seen.sum()), float(rr_t[seen].mean())),
                 (f"TGN_seed{s}", "novel", int((~seen).sum()), float(rr_t[~seen].mean()))]
with open(f"{REPO_DIR}/results/rq2_slices_wiki.csv", "w") as f:
    f.write("model,slice,n_queries,mrr\n")
    for r in rows: f.write(",".join(map(str, r)) + "\n")
save_and_push("RQ2 recurring/novel slices (query-time definition)", ["results/rq2_slices_wiki.csv"])

surprise index, my computation: 0.394   (TGB paper: 0.108)
tw_default_rho0.15 recurring share 0.8382 | MRR rec 0.6809 | MRR novel 0.002
full_R-EdgeBank recurring share 0.8382 | MRR rec 0.8643 | MRR novel 0.002

A  results/rq2_slices_wiki.csv

[main 7966915] RQ2 recurring/novel slices (query-time definition)
 1 file changed, 5 insertions(+)
 create mode 100644 results/rq2_slices_wiki.csv

Current branch main is up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/Temporal-Graph-ML-Project.git
   3661cb3..7966915  main -> main



In [10]:
!ls {REPO_DIR}/third_party/TGB/examples/linkproppred/tgbl-review/
!python {REPO_DIR}/third_party/TGB/examples/linkproppred/tgbl-wiki/tgn.py --help | head -40

dyrep.py  edgebank.py  tgn.py
Traceback (most recent call last):
  File "/kaggle/working/Temporal-Graph-ML-Project/third_party/TGB/examples/linkproppred/tgbl-wiki/tgn.py", line 29, in <module>
    from modules.decoder import LinkPredictor
ModuleNotFoundError: No module named 'modules'


In [11]:
TGB = f"{REPO_DIR}/third_party/TGB"
!cd {TGB} && PYTHONPATH=. python examples/linkproppred/tgbl-review/tgn.py --help | head -40
!ls {REPO_DIR}/logs

Traceback (most recent call last):
  File "/kaggle/working/Temporal-Graph-ML-Project/third_party/TGB/examples/linkproppred/tgbl-review/tgn.py", line 33, in <module>
    from modules.msg_agg import LastAggregator
  File "/kaggle/working/Temporal-Graph-ML-Project/third_party/TGB/modules/msg_agg.py", line 12, in <module>
    from torch_scatter import scatter_max
ModuleNotFoundError: No module named 'torch_scatter'
archive_graphmixer_1epoch    graphmixer_source.txt
dyrep_wiki_seed1.txt	     graphmixer_wiki_seed1.txt
dyrep_wiki_seed2.txt	     graphmixer_wiki_seed2.txt
dyrep_wiki_seed3.txt	     graphmixer_wiki_seed3.txt
edgebank_inf_wiki_seed1.txt  graphmixer_wiki_seed4.txt
edgebank_inf_wiki_seed2.txt  hardware_session2.txt
edgebank_inf_wiki_seed3.txt  hardware.txt
edgebank_tw_wiki_seed1.txt   requirements_session2.txt
edgebank_tw_wiki_seed2.txt   tgn_wiki_seed1.txt
edgebank_tw_wiki_seed3.txt   tgn_wiki_seed2.txt
edgebank_tw_wiki.txt	     tgn_wiki_seed3.txt
edgebank_wiki.txt	     versions_se

In [12]:
!grep -iE "torch|scatter|sparse|cluster|geometric|tgb|python" {REPO_DIR}/logs/versions.txt
!echo "--- session2 ---"
!grep -iE "torch|scatter|sparse|geometric" {REPO_DIR}/logs/requirements_session2.txt
!echo "--- first lines of the wiki TGN log ---"
!head -5 {REPO_DIR}/logs/tgn_wiki_seed1.txt
!python -c "import torch; print('now:', torch.__version__, torch.version.cuda)"

torch 2.11.0+cu128
--- session2 ---
nvidia-cusparse-cu12==12.5.8.93
nvidia-cusparselt-cu12==0.7.1
pytorch-ignite==0.5.5
pytorch-lightning==2.6.6
torch==2.11.0+cu128
torch-geometric==2.8.1
torch_scatter==2.1.2+pt211cu128
torchao==0.10.0
torchaudio==2.11.0+cu128
torchcodec==0.11.0+cu128
torchdata==0.11.0
torchinfo==1.8.0
torchmetrics==1.9.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision==0.26.0+cu128
--- first lines of the wiki TGN log ---
/kaggle/working/ML_Term_Project/third_party/TGB/examples/linkproppred/tgbl-wiki/tgn.py:168: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  "y_pred_neg": np.array(y_pred[1:, :].squeeze(dim=-1).cpu()),
INFO: Arguments: Namespace(data='tgbl-wiki', lr=0.0001, bs=200, k_value=10, num_epoch=30, seed=1, mem_dim=100, time_dim=100, emb_dim=100, tolerance=1e-06, patience=5, num_run=1)
=================*** TGN: LinkPropPred: tgbl-wiki ***=

In [13]:
import torch, subprocess
r = subprocess.run("pip install -q torch_scatter -f https://data.pyg.org/whl/torch-2.11.0+cu130.html",
                   shell=True, capture_output=True, text=True)
print((r.stdout + r.stderr)[-600:])
try:
    from torch_scatter import scatter_max
    x = torch.randn(10, device="cuda"); idx = torch.zeros(10, dtype=torch.long, device="cuda")
    print("torch_scatter OK on GPU:", scatter_max(x, idx)[0].shape)
except Exception as e:
    print("FAILED:", repr(e)[:300])


torch_scatter OK on GPU: torch.Size([1])


In [15]:
TGB = f"{REPO_DIR}/third_party/TGB"
!cd {TGB} && PYTHONPATH=. python examples/linkproppred/tgbl-review/tgn.py --help | head -40

usage: *** TGB *** [-h] [-d DATA] [--lr LR] [--bs BS] [--k_value K_VALUE]
                   [--num_epoch NUM_EPOCH] [--seed SEED] [--mem_dim MEM_DIM]
                   [--time_dim TIME_DIM] [--emb_dim EMB_DIM]
                   [--tolerance TOLERANCE] [--patience PATIENCE]
                   [--num_run NUM_RUN]

options:
  -h, --help            show this help message and exit
  -d, --data DATA       Dataset name
  --lr LR               Learning rate
  --bs BS               Batch size
  --k_value K_VALUE     k_value for computing ranking metrics
  --num_epoch NUM_EPOCH
                        Number of epochs
  --seed SEED           Random seed
  --mem_dim MEM_DIM     Memory dimension
  --time_dim TIME_DIM   Time dimension
  --emb_dim EMB_DIM     Embedding dimension
  --tolerance TOLERANCE
                        Early stopper tolerance
  --patience PATIENCE   Early stopper patience
  --num_run NUM_RUN     Number of iteration runs
usage: *** TGB *** [-h] [-d DATA] [--lr LR] [--bs BS]

In [16]:
!grep -iE "elapsed|time|epoch|mrr" {REPO_DIR}/logs/tgn_wiki_seed1.txt | tail -25

	Validation: Elapsed time (s):  75.4732
Epoch: 13, Loss: 1.1185, Training elapsed Time (s):  12.4131
	Validation mrr:  0.3874
	Validation: Elapsed time (s):  75.4160
Epoch: 14, Loss: 1.1155, Training elapsed Time (s):  12.1963
	Validation mrr:  0.3675
	Validation: Elapsed time (s):  75.0973
Epoch: 15, Loss: 1.1166, Training elapsed Time (s):  12.2418
	Validation mrr:  0.3289
	Validation: Elapsed time (s):  75.5368
Epoch: 16, Loss: 1.1150, Training elapsed Time (s):  12.5614
	Validation mrr:  0.3556
	Validation: Elapsed time (s):  75.1407
Epoch: 17, Loss: 1.1156, Training elapsed Time (s):  12.2841
	Validation mrr:  0.3419
	Validation: Elapsed time (s):  75.4500
Epoch: 18, Loss: 1.1137, Training elapsed Time (s):  12.2888
	Validation mrr:  0.3529
	Validation: Elapsed time (s):  75.2314
Train & Validation: Elapsed Time (s):  1592.3976
INFO: load the model of epoch 13 from /kaggle/working/ML_Term_Project/third_party/TGB/examples/linkproppred/tgbl-wiki/saved_models//TGN_tgbl-wiki_1_0.pth
	

In [17]:
import subprocess, time
TGB = f"{REPO_DIR}/third_party/TGB"
EPOCHS_CAP = 30            # script default; lower it if one epoch is too slow (see step 1)
SEED = 1
LOG = f"{REPO_DIR}/logs/tgn_review_seed{SEED}.txt"
cmd = (f"cd {TGB} && CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python examples/linkproppred/tgbl-review/tgn.py "
       f"--data tgbl-review --seed {SEED} --num_epoch {EPOCHS_CAP} > {LOG} 2>&1")
proc = subprocess.Popen(cmd, shell=True)
print("started at", time.strftime("%H:%M:%S"))

started at 05:11:56


In [18]:
print("running" if proc.poll() is None else f"ended, exit {proc.returncode}")
!tr '\r' '\n' < {LOG} | grep -v '^$' | tail -6
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

running
index, utilization.gpu [%], memory.used [MiB]
0, 0 %, 123 MiB
1, 0 %, 3 MiB


In [19]:
TGB = f"{REPO_DIR}/third_party/TGB"
for tag, flag in [("inf", ""), ("tw", "--mem_mode fixed_time_window")]:
    lp = f"{REPO_DIR}/logs/edgebank_{tag}_review_seed1.txt"
    if not os.path.exists(lp):
        !cd {TGB} && PYTHONPATH=. python examples/linkproppred/tgbl-review/edgebank.py --data tgbl-review --seed 1 {flag} 2>&1 | tee {lp}
save_and_push("EdgeBank on tgbl-review", ["logs"])

============*** EdgeBank: unlimited: tgbl-review ***==============
100%|██████████| 3654/3654 [02:25<00:00, 25.16it/s]
INFO: val: Evaluation Setting: >>> ONE-VS-MANY <<< 
	val: mrr:  0.0229
	val: Elapsed Time (s):  145.2777
100%|██████████| 3645/3645 [02:27<00:00, 24.78it/s]
INFO: Test: Evaluation Setting: >>> ONE-VS-MANY <<< 
	Test: mrr:  0.0229
	Test: Elapsed Time (s):  147.1703
============*** EdgeBank: fixed_time_window: tgbl-review ***==============
100%|██████████| 3654/3654 [02:27<00:00, 24.70it/s]
INFO: val: Evaluation Setting: >>> ONE-VS-MANY <<< 
	val: mrr:  0.0242
	val: Elapsed Time (s):  148.0224
100%|██████████| 3645/3645 [02:29<00:00, 24.34it/s]
INFO: Test: Evaluation Setting: >>> ONE-VS-MANY <<< 
	Test: mrr:  0.0253
	Test: Elapsed Time (s):  149.8274

A  logs/edgebank_inf_review_seed1.txt
A  logs/edgebank_tw_review_seed1.txt
A  logs/tgn_review_seed1.txt

[main e834285] EdgeBank on tgbl-review
 3 files changed, 25 insertions(+)
 create mode 100644 logs/edgebank_inf_review

In [20]:
dsr, evr = redgebank.load("tgbl-review", f"{REPO_DIR}/third_party/TGB")
N = 100_000                                  # first N validation edges only, to keep it affordable
rows = []
for rho in [0.05, 0.15, 0.40]:
    for mode, k in [("tw", 1.0), ("recency", 1.0), ("full", 1.0), ("full", 4.0)]:
        r = redgebank.evaluate(dsr, evr, "val", rho, k, mode, max_edges=N)
        rows.append((mode, rho, k, r["mrr"], r["secs"])); print(mode, rho, k, round(r["mrr"], 4))
with open(f"{REPO_DIR}/results/sweep_val_review.csv", "w") as f:
    f.write("mode,rho,kappa,val_mrr_first_100k,secs\n")
    for r in rows: f.write(",".join(map(str, r)) + "\n")
b = {m: max([r for r in rows if r[0] == m], key=lambda r: r[3]) for m in ["tw", "recency", "full"]}
test_out = []
for name, m in [("tw_tuned", "tw"), ("recency_only", "recency"), ("full_R-EdgeBank", "full")]:
    r = redgebank.evaluate(dsr, evr, "test", b[m][1], b[m][2], m)
    test_out.append((name, m, b[m][1], b[m][2], r["mrr"], r["secs"])); print(name, round(r["mrr"], 4))
with open(f"{REPO_DIR}/results/test_final_review.csv", "w") as f:
    f.write("config,mode,rho,kappa,test_mrr,test_secs\n")
    for o in test_out: f.write(",".join(map(str, o)) + "\n")
save_and_push("R-EdgeBank on tgbl-review", ["results"])

tw 0.05 1.0 0.0261
recency 0.05 1.0 0.0303
full 0.05 1.0 0.0295
full 0.05 4.0 0.0295
tw 0.15 1.0 0.0235
recency 0.15 1.0 0.0298
full 0.15 1.0 0.0295
full 0.15 4.0 0.0295
tw 0.4 1.0 0.0226
recency 0.4 1.0 0.0295
full 0.4 1.0 0.0295
full 0.4 4.0 0.0295
tw_tuned 0.0293
recency_only 0.0293
full_R-EdgeBank 0.0284

 M logs/tgn_review_seed1.txt
A  results/sweep_val_review.csv
A  results/test_final_review.csv

[main 3d36290] R-EdgeBank on tgbl-review
 2 files changed, 17 insertions(+)
 create mode 100644 results/sweep_val_review.csv
 create mode 100644 results/test_final_review.csv

error: cannot pull with rebase: You have unstaged changes.
error: Please commit or stash them.

To https://github.com/Rosierobber/Temporal-Graph-ML-Project.git
   e834285..3d36290  main -> main



In [21]:
d = ds.full_data; S, D = d["sources"], d["destinations"]
tr, va, te = ds.train_mask, ds.val_mask, ds.test_mask
te_pairs = list(zip(S[te].tolist(), D[te].tolist()))
for name, m in [("train only", tr), ("train+val", tr | va)]:
    seen = set(zip(S[m].tolist(), D[m].tolist()))
    print(f"{name:10s} per interaction: {np.mean([p not in seen for p in te_pairs]):.3f}   "
          f"unique pairs: {np.mean([p not in seen for p in set(te_pairs)]):.3f}")

train only per interaction: 0.394   unique pairs: 0.617
train+val  per interaction: 0.291   unique pairs: 0.554


In [22]:
!ps aux | grep "tgbl-review/tgn.py" | grep -v grep | head -2
!tr '\r' '\n' < {REPO_DIR}/logs/tgn_review_seed1.txt | grep -v '^$' | tail -8


root         419  0.0  0.0   2816  1968 ?        S    05:11   0:00 /bin/sh -c cd /kaggle/working/Temporal-Graph-ML-Project/third_party/TGB && CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python examples/linkproppred/tgbl-review/tgn.py --data tgbl-review --seed 1 --num_epoch 30 > /kaggle/working/Temporal-Graph-ML-Project/logs/tgn_review_seed1.txt 2>&1
root         420 98.9 16.8 20595764 5524164 ?    Rl   05:11  55:28 python3 examples/linkproppred/tgbl-review/tgn.py --data tgbl-review --seed 1 --num_epoch 30
4716112it [00:14, 335394.57it/s]
4749655it [00:14, 334516.13it/s]
4783110it [00:14, 334279.20it/s]
4816540it [00:14, 333053.37it/s]
4850141it [00:15, 333933.32it/s]
4873541it [00:15, 322165.66it/s]
/kaggle/working/Temporal-Graph-ML-Project/third_party/TGB/examples/linkproppred/tgbl-review/tgn.py:169: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  "y_pred_neg": np.array(y

In [23]:
!grep -n "perf_list\|def test\|evaluator.eval\|perf_metrics\|split_mode" {REPO_DIR}/third_party/TGB/examples/linkproppred/tgbl-wiki/tgn.py

111:def test(loader, neg_sampler, split_mode):
119:        split_mode: specifies whether it is the 'validation' or 'test' set to correctly load the negatives
127:    perf_list = []
137:        neg_batch_list = neg_sampler.query_batch(pos_src, pos_dst, pos_t, split_mode=split_mode)
171:            perf_list.append(evaluator.eval(input_dict)[metric])
177:    perf_metrics = float(torch.tensor(perf_list).mean())
179:    return perf_metrics
305:    val_perf_list = []
317:        perf_metric_val = test(val_loader, neg_sampler, split_mode="val")
320:        val_perf_list.append(perf_metric_val)
338:    perf_metric_test = test(test_loader, neg_sampler, split_mode="test")
349:                  f'val {metric}': val_perf_list,


In [25]:
import csv, numpy as np, matplotlib.pyplot as plt
os.makedirs(f"{REPO_DIR}/figures", exist_ok=True)
rd = lambda p: list(csv.DictReader(open(f"{REPO_DIR}/{p}")))

def save_and_push(msg, paths=("notes", "results", "logs", "README.md")):
    git("add", *paths); git("status", "--short"); git("commit", "-m", msg)
    git("pull", "--rebase", "--autostash", "origin", "main")   # --autostash: tolerates the running TGN log
    git("push", "origin", "main")

sw = rd("results/sweep_val_wiki.csv")
rhos = sorted({float(r["rho"]) for r in sw})
best = lambda mode, a: max(float(r["val_mrr"]) for r in sw if r["mode"] == mode and float(r["rho"]) == a)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(rhos, [best("tw", a) for a in rhos], marker="o", label="EdgeBank (window)")
ax.plot(rhos, [best("recency", a) for a in rhos], marker="s", label="recency inside window")
ax.axhline(max(float(r["val_mrr"]) for r in sw if r["mode"] == "full"), ls="--", color="k", label="R-EdgeBank (no window)")
ax.set_xlabel("window ratio rho"); ax.set_ylabel("Val MRR"); ax.legend()
plt.tight_layout(); plt.savefig(f"{REPO_DIR}/figures/fig3_sensitivity_wiki.png", dpi=300); plt.close()
save_and_push("Fig 3: sensitivity to window ratio", ["figures"])


A  figures/fig3_sensitivity_wiki.png
 M logs/tgn_review_seed1.txt

[main 99c3109] Fig 3: sensitivity to window ratio
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 figures/fig3_sensitivity_wiki.png

Created autostash: 5d4040b
Current branch main is up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD
Applied autostash.

To https://github.com/Rosierobber/Temporal-Graph-ML-Project.git
   3d36290..99c3109  main -> main



In [26]:
!tr '\r' '\n' < {REPO_DIR}/logs/tgn_review_seed1.txt | grep -E "Epoch|mrr|Elapsed" | tail -6
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

index, utilization.gpu [%], memory.used [MiB]
0, 36 %, 6199 MiB
1, 0 %, 3 MiB


In [27]:
TGB = f"{REPO_DIR}/third_party/TGB"
src = f"{TGB}/examples/linkproppred/tgbl-wiki/tgn.py"
dst = f"{REPO_DIR}/scripts/tgn_wiki_perquery.py"
code = open(src).read()
old = "    perf_metrics = float(torch.tensor(perf_list).mean())"
assert code.count(old) == 1, "pattern not found exactly once; paste lines 170-180 of tgn.py"
new = ("    if split_mode == 'test':  # ADDED (own change): save per-query reciprocal ranks for the RQ2 analysis\n"
       "        import os as _os\n"
       "        np.save(_os.environ.get('PERQUERY_OUT', 'perquery_test.npy'), np.array(perf_list))\n" + old)
open(dst, "w").write(code.replace(old, new))
print(subprocess.run(["diff", src, dst], capture_output=True, text=True).stdout)

176a177,179
>     if split_mode == 'test':  # ADDED (own change): save per-query reciprocal ranks for the RQ2 analysis
>         import os as _os
>         np.save(_os.environ.get('PERQUERY_OUT', 'perquery_test.npy'), np.array(perf_list))



In [28]:
os.makedirs(f"{REPO_DIR}/results/perquery", exist_ok=True)
def one(s):
    out = f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed{s}.npy"
    log = f"{REPO_DIR}/logs/tgn_wiki_perquery_seed{s}.txt"
    return (f"cd {TGB} && CUDA_VISIBLE_DEVICES=1 PERQUERY_OUT={out} PYTHONPATH=. "
            f"python {dst} --data tgbl-wiki --seed {s} > {log} 2>&1")
proc2 = subprocess.Popen(" ; ".join(one(s) for s in [1, 2, 3]), shell=True)
print("started", time.strftime("%H:%M:%S"))

started 06:25:41


In [29]:
!ps aux | grep "tgn_wiki_perquery" | grep -v grep | head -3
!ls -la {REPO_DIR}/logs/tgn_wiki_perquery_seed*.txt
!tr '\r' '\n' < {REPO_DIR}/logs/tgn_wiki_perquery_seed1.txt | grep -v '^$' | tail -6
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv

root         587  0.0  0.0   2816  1884 ?        S    06:25   0:00 /bin/sh -c cd /kaggle/working/Temporal-Graph-ML-Project/third_party/TGB && CUDA_VISIBLE_DEVICES=1 PERQUERY_OUT=/kaggle/working/Temporal-Graph-ML-Project/results/perquery/tgn_wiki_test_seed1.npy PYTHONPATH=. python /kaggle/working/Temporal-Graph-ML-Project/scripts/tgn_wiki_perquery.py --data tgbl-wiki --seed 1 > /kaggle/working/Temporal-Graph-ML-Project/logs/tgn_wiki_perquery_seed1.txt 2>&1 ; cd /kaggle/working/Temporal-Graph-ML-Project/third_party/TGB && CUDA_VISIBLE_DEVICES=1 PERQUERY_OUT=/kaggle/working/Temporal-Graph-ML-Project/results/perquery/tgn_wiki_test_seed2.npy PYTHONPATH=. python /kaggle/working/Temporal-Graph-ML-Project/scripts/tgn_wiki_perquery.py --data tgbl-wiki --seed 2 > /kaggle/working/Temporal-Graph-ML-Project/logs/tgn_wiki_perquery_seed2.txt 2>&1 ; cd /kaggle/working/Temporal-Graph-ML-Project/third_party/TGB && CUDA_VISIBLE_DEVICES=1 PERQUERY_OUT=/kaggle/working/Temporal-Graph-ML-Project/results/perq

In [30]:
import re, time, subprocess
from IPython.display import clear_output

LOGS = {
    "TGN review  seed1":      f"{REPO_DIR}/logs/tgn_review_seed1.txt",
    "TGN wiki per-query s1":  f"{REPO_DIR}/logs/tgn_wiki_perquery_seed1.txt",
    "TGN wiki per-query s2":  f"{REPO_DIR}/logs/tgn_wiki_perquery_seed2.txt",
    "TGN wiki per-query s3":  f"{REPO_DIR}/logs/tgn_wiki_perquery_seed3.txt",
}

def summarize(path):
    if not os.path.exists(path): return "no log yet"
    t = open(path, errors="ignore").read().replace("\r", "\n")
    ep   = re.findall(r"Epoch:\s*(\d+), Loss:\s*([0-9.]+), Training elapsed Time \(s\):\s*([0-9.]+)", t)
    val  = re.findall(r"Validation mrr:\s*([0-9.]+)", t)
    vt   = re.findall(r"Validation: Elapsed time \(s\):\s*([0-9.]+)", t)
    test = re.findall(r"Test: mrr:\s*([0-9.]+)", t)
    if "Traceback" in t: return "ERROR: " + t.strip().splitlines()[-1][:120]
    s = f"epochs done: {len(ep)}"
    if ep:  s += f" | loss {ep[-1][1]}"
    if val: s += f" | val MRR {val[-1]} (best {max(map(float, val)):.4f})"
    if ep and vt: s += f" | ~{float(ep[-1][2]) + float(vt[-1]):.0f}s/epoch"
    if test: s += f" | TEST MRR {test[-1]}  <- finished"
    return s

def n_running(pattern):
    return int(subprocess.run(f'ps aux | grep "[{pattern[0]}]{pattern[1:]}" | wc -l', shell=True,
                              capture_output=True, text=True).stdout.strip() or 0)

def dashboard(minutes=25, every=60):
    t0 = time.time()
    while time.time() - t0 < minutes * 60:
        clear_output(wait=True)
        print(time.strftime("%H:%M:%S"), f"| monitor stops after {minutes} min, re-run it to continue. Do NOT press Stop.")
        print("review TGN process:", "RUNNING" if n_running("tgbl-review/tgn.py") else "not running",
              "| wiki per-query process:", "RUNNING" if n_running("tgn_wiki_perquery") else "not running", "\n")
        for k, p in LOGS.items(): print(f"{k:24s} {summarize(p)}")
        print("\n" + subprocess.run("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                                    shell=True, capture_output=True, text=True).stdout)
        time.sleep(every)

dashboard()

06:58:16 | monitor stops after 25 min, re-run it to continue. Do NOT press Stop.
review TGN process: RUNNING | wiki per-query process: RUNNING 

TGN review  seed1        epochs done: 0
TGN wiki per-query s1    epochs done: 0
TGN wiki per-query s2    no log yet
TGN wiki per-query s3    no log yet

0, 24 %, 6199 MiB
1, 41 %, 555 MiB



In [31]:
import re, time, os
print(time.strftime("%H:%M:%S"), "| review alive:", proc.poll() is None, "| wiki per-query alive:", proc2.poll() is None)
for s in [1, 2, 3]:
    f = f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed{s}.npy"
    print(f"wiki seed {s}:", "done" if os.path.exists(f) else "not finished")

06:59:16 | review alive: True | wiki per-query alive: True
wiki seed 1: done
wiki seed 2: not finished
wiki seed 3: not finished


In [32]:
import numpy as np, re, os
rr = np.load(f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed1.npy")
seen = np.load(f"{REPO_DIR}/results/perquery_test_wiki_tw_default_rho0.15.npz")["seen"]
print("queries saved:", len(rr), "| expected:", len(seen), "| match:", len(rr) == len(seen))

t = open(f"{REPO_DIR}/logs/tgn_wiki_perquery_seed1.txt", errors="ignore").read().replace("\r", "\n")
log_test = re.findall(r"Test: mrr:\s*([0-9.]+)", t)
print("overall MRR from per-query file:", round(float(rr.mean()), 4), "| 'Test: mrr' in log:", log_test[-1] if log_test else "not found")

if len(rr) == len(seen):
    print(f"recurring ({seen.mean():.2%} of queries): TGN {rr[seen].mean():.4f}   (EdgeBank 0.6809, R-EdgeBank 0.8643)")
    print(f"novel     ({(~seen).mean():.2%} of queries): TGN {rr[~seen].mean():.4f}   (EdgeBank 0.0020, R-EdgeBank 0.0020)")

queries saved: 23621 | expected: 23621 | match: True
overall MRR from per-query file: 0.3273 | 'Test: mrr' in log: not found
recurring (83.82% of queries): TGN 0.3869   (EdgeBank 0.6809, R-EdgeBank 0.8643)
novel     (16.18% of queries): TGN 0.0186   (EdgeBank 0.0020, R-EdgeBank 0.0020)


In [33]:
import shutil, os, time, subprocess
src = f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed1.npy"
keep = f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed1_SAFE.npy"
shutil.copy2(src, keep)
print("file written:", time.ctime(os.path.getmtime(src)), "| now:", time.ctime())
print(subprocess.run("ps aux | grep '[t]gn_wiki_perquery.py' | awk '{print $2, $12, $13, $14, $15, $16, $17}'",
                     shell=True, capture_output=True, text=True).stdout)
!grep -n "num_run\|SEED" {REPO_DIR}/scripts/tgn_wiki_perquery.py | head -12

file written: Sun Nov 22 06:56:50 2026 | now: Sun Nov 22 07:11:25 2026
587 -c cd /kaggle/working/Temporal-Graph-ML-Project/third_party/TGB && CUDA_VISIBLE_DEVICES=1 PERQUERY_OUT=/kaggle/working/Temporal-Graph-ML-Project/results/perquery/tgn_wiki_test_seed1.npy
588 /kaggle/working/Temporal-Graph-ML-Project/scripts/tgn_wiki_perquery.py --data tgbl-wiki --seed 1 

7:    python examples/linkproppred/tgbl-wiki/tgn.py --data "tgbl-wiki" --num_run 1 --seed 1
202:SEED = args.seed
208:NUM_RUNS = args.num_run
259:    torch.manual_seed(run_idx + SEED)
260:    set_random_seed(run_idx + SEED)
300:    save_model_id = f'{MODEL_NAME}_{DATA}_{SEED}_{run_idx}'
351:                  'seed': SEED,


In [34]:
!pkill -f "[t]gn_wiki_perquery"
time.sleep(3)
shutil.copy2(keep, src)       # restore the seed 1 file, in case a later run overwrote it
script = f"{REPO_DIR}/scripts/tgn_wiki_perquery.py"

def run_perquery(seed, gpu=1):
    out = f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed{seed}.npy"
    log = f"{REPO_DIR}/logs/tgn_wiki_perquery_seed{seed}.txt"
    return (f"cd {TGB} && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} PERQUERY_OUT={out} PYTHONPATH=. "
            f"python -u {script} --data tgbl-wiki --seed {seed} --num_run 1 > {log} 2>&1")

proc2 = subprocess.Popen(run_perquery(2) + " ; " + run_perquery(3), shell=True, start_new_session=True)
print("seeds 2 and 3 started", time.strftime("%H:%M:%S"))

seeds 2 and 3 started 07:11:45


In [35]:
import re, numpy as np
print("chain alive:", proc2.poll() is None, "|", time.strftime("%H:%M:%S"))
for s in [2, 3]:
    p = f"{REPO_DIR}/logs/tgn_wiki_perquery_seed{s}.txt"
    t = open(p, errors="ignore").read().replace("\r", "\n") if os.path.exists(p) else ""
    ep, val, test = (re.findall(x, t) for x in [r"Epoch:\s*(\d+)", r"Validation mrr:\s*([0-9.]+)", r"Test: mrr:\s*([0-9.]+)"])
    print(f"seed {s}:", f"epoch {ep[-1]}" if ep else "not started", f"| last val {val[-1]}" if val else "", f"| TEST {test[-1]}" if test else "")
a = np.load(f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed1.npy")
b = np.load(f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed1_SAFE.npy")
print("seed 1 file intact:", np.array_equal(a, b), "| mean", round(float(a.mean()), 4))

chain alive: True | 07:13:26
seed 2: epoch 01  
seed 3: not started  
seed 1 file intact: True | mean 0.3273


In [ ]:
# ==============================================================================
# ALL-IN-ONE PIPELINE: Setup Dependencies -> Wiki TGN (1-3) -> Review TGN -> RQ2
# ==============================================================================
import os, sys, time, subprocess, re, json
import numpy as np
import torch

# ------------------------------------------------------------------------------
# 0. Install & Verify torch_scatter on GPU (Prevents ModuleNotFoundError)
# ------------------------------------------------------------------------------
print("⏳ Checking and installing torch_scatter...")
wheel_url = f"https://data.pyg.org/whl/torch-{torch.__version__}.html"
install_cmd = (
    f"pip install -q torch_scatter -f {wheel_url} "
    f"-f https://data.pyg.org/whl/torch-2.11.0+cu130.html"
)
subprocess.run(install_cmd, shell=True, check=False)

try:
    from torch_scatter import scatter_max
    x = torch.randn(10, device="cuda" if torch.cuda.is_available() else "cpu")
    idx = torch.zeros(10, dtype=torch.long, device=x.device)
    scatter_max(x, idx)
    print("✅ torch_scatter successfully loaded on GPU!")
except Exception as e:
    print(f"⚠️ Warning: torch_scatter test failed ({e}), installing directly...")
    subprocess.run("pip install -q torch_scatter", shell=True)

# ------------------------------------------------------------------------------
# 1. Ensure scripts/tgn_wiki_perquery.py exists with per-query rank hook
# ------------------------------------------------------------------------------
TGB = f"{REPO_DIR}/third_party/TGB"
os.makedirs(f"{REPO_DIR}/results/perquery", exist_ok=True)
os.makedirs(f"{REPO_DIR}/logs", exist_ok=True)
os.makedirs(f"{REPO_DIR}/scripts", exist_ok=True)

src = f"{TGB}/examples/linkproppred/tgbl-wiki/tgn.py"
dst = f"{REPO_DIR}/scripts/tgn_wiki_perquery.py"

code = open(src).read()
old = "    perf_metrics = float(torch.tensor(perf_list).mean())"
patch = (
    "    if split_mode == 'test':  # Save per-query reciprocal ranks for RQ2\n"
    "        import os as _os, numpy as _np\n"
    "        _out_path = _os.environ.get('PERQUERY_OUT', 'perquery_test.npy')\n"
    "        _os.makedirs(_os.path.dirname(_os.path.abspath(_out_path)), exist_ok=True)\n"
    "        _np.save(_out_path, _np.array(perf_list))\n"
    "        print(f'>>> SAVED PER-QUERY TEST ARRAY ({len(perf_list)} queries) to {_out_path}')\n"
    + old
)

if code.count(old) == 1:
    open(dst, "w").write(code.replace(old, patch))
    print("✅ scripts/tgn_wiki_perquery.py prepared!")
else:
    print("ℹ️ scripts/tgn_wiki_perquery.py already patched.")

# ------------------------------------------------------------------------------
# 2. Define the tasks queue
# ------------------------------------------------------------------------------
TASKS = [
    {
        "name": "Wiki TGN per-query (Seed 1)",
        "cmd": f"cd {TGB} && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=1 PERQUERY_OUT={REPO_DIR}/results/perquery/tgn_wiki_test_seed1.npy PYTHONPATH=. python -u {dst} --data tgbl-wiki --seed 1 --num_run 1",
        "log": f"{REPO_DIR}/logs/tgn_wiki_perquery_seed1.txt",
        "target": f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed1.npy",
        "git_paths": ["results/perquery/tgn_wiki_test_seed1.npy", "logs/tgn_wiki_perquery_seed1.txt"],
        "weight": 1
    },
    {
        "name": "Wiki TGN per-query (Seed 2)",
        "cmd": f"cd {TGB} && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=1 PERQUERY_OUT={REPO_DIR}/results/perquery/tgn_wiki_test_seed2.npy PYTHONPATH=. python -u {dst} --data tgbl-wiki --seed 2 --num_run 1",
        "log": f"{REPO_DIR}/logs/tgn_wiki_perquery_seed2.txt",
        "target": f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed2.npy",
        "git_paths": ["results/perquery/tgn_wiki_test_seed2.npy", "logs/tgn_wiki_perquery_seed2.txt"],
        "weight": 1
    },
    {
        "name": "Wiki TGN per-query (Seed 3)",
        "cmd": f"cd {TGB} && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=1 PERQUERY_OUT={REPO_DIR}/results/perquery/tgn_wiki_test_seed3.npy PYTHONPATH=. python -u {dst} --data tgbl-wiki --seed 3 --num_run 1",
        "log": f"{REPO_DIR}/logs/tgn_wiki_perquery_seed3.txt",
        "target": f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed3.npy",
        "git_paths": ["results/perquery/tgn_wiki_test_seed3.npy", "logs/tgn_wiki_perquery_seed3.txt"],
        "weight": 1
    },
    {
        "name": "Review TGN (Seed 1, 30 epochs)",
        "cmd": f"cd {TGB} && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES=0 PYTHONPATH=. python -u examples/linkproppred/tgbl-review/tgn.py --data tgbl-review --seed 1 --num_epoch 30",
        "log": f"{REPO_DIR}/logs/tgn_review_seed1.txt",
        "target": f"{REPO_DIR}/logs/tgn_review_seed1.txt",
        "git_paths": ["logs/tgn_review_seed1.txt", "results"],
        "weight": 3
    }
]

total_weight = sum(t["weight"] for t in TASKS)
completed_weight = 0
start_time = time.time()

def print_progress(current_idx, current_name):
    pct = (completed_weight / total_weight) * 100.0
    bar_len = 30
    filled = int(bar_len * (completed_weight / total_weight))
    bar = "█" * filled + "░" * (bar_len - filled)
    elapsed = time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))
    print("\n" + "=" * 78)
    print(f"📊 OVERALL PIPELINE PROGRESS: [{bar}] {pct:.1f}% COMPLETE (Elapsed: {elapsed})")
    print(f"▶ Currently Running [{current_idx+1}/{len(TASKS)}]: {current_name}")
    print("=" * 78 + "\n")

# ------------------------------------------------------------------------------
# 3. Main Sequential Execution Loop
# ------------------------------------------------------------------------------
for idx, task in enumerate(TASKS):
    print_progress(idx, task["name"])
    
    # Check if task is already finished
    if os.path.exists(task["target"]) and os.path.getsize(task["target"]) > 0:
        if "perquery" in task["target"]:
            try:
                arr = np.load(task["target"])
                if len(arr) == 23621:
                    print(f"⏩ {task['name']} is ALREADY COMPLETE ({len(arr)} queries, MRR: {arr.mean():.4f}). Skipping to next...")
                    completed_weight += task["weight"]
                    continue
            except Exception:
                pass
        elif "review" in task["target"]:
            content = open(task["target"], errors="ignore").read()
            if "Test: mrr" in content:
                print(f"⏩ {task['name']} is ALREADY COMPLETE. Skipping to next...")
                completed_weight += task["weight"]
                continue

    # Execute and stream live directly to screen
    with open(task["log"], "w", encoding="utf-8") as f_log:
        proc = subprocess.Popen(
            task["cmd"], shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        for line in iter(proc.stdout.readline, ''):
            if line:
                sys.stdout.write(line)
                sys.stdout.flush()
                f_log.write(line)
                f_log.flush()
        proc.stdout.close()
        ret = proc.wait()

    if ret == 0 and os.path.exists(task["target"]):
        print(f"\n🎉 {task['name']} FINISHED SUCCESSFULLY!")
        save_and_push(f"Complete {task['name']}", task["git_paths"])
        completed_weight += task["weight"]
    else:
        print(f"\n❌ {task['name']} FAILED with exit code {ret}.")
        # Print the last 15 lines of log for immediate diagnosis
        if os.path.exists(task["log"]):
            print("--- Log Tail ---")
            lines = open(task["log"], errors="ignore").readlines()
            print("".join(lines[-15:]))
        break

# Final progress bar (100%)
bar = "█" * 30
elapsed = time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))
print("\n" + "=" * 78)
print(f"🏁 ALL TASKS COMPLETED! [{bar}] 100.0% (Total Time: {elapsed})")
print("=" * 78 + "\n")

# ------------------------------------------------------------------------------
# 4. Harvest & Print Final Results Table
# ------------------------------------------------------------------------------
print("=" * 78)
print("📈 FINAL SUMMARY RESULTS")
print("=" * 78)

# A. Wiki RQ2 Recurring vs Novel Breakdown
try:
    from tgb.linkproppred.dataset import LinkPropPredDataset
    ds = LinkPropPredDataset(name="tgbl-wiki", root="datasets")
    d = ds.full_data
    tr, va, te = ds.train_mask, ds.val_mask, ds.test_mask
    history = set(zip(d['sources'][tr | va].tolist(), d['destinations'][tr | va].tolist()))
    test_pairs = list(zip(d['sources'][te].tolist(), d['destinations'][te].tolist()))
    seen = np.array([p in history for p in test_pairs])
    
    print("\n[tgbl-wiki: TGN RQ2 Recurring vs. Novel Slices]")
    print(f"Total Test Interactions: {len(seen):,} | Recurring: {seen.sum():,} ({seen.mean():.2%}) | Novel: {(~seen).sum():,} ({(~seen).mean():.2%})\n")
    print(f"{'Seed':<8} | {'Overall MRR':<14} | {'Recurring MRR':<16} | {'Novel MRR':<14}")
    print("-" * 60)
    for s in [1, 2, 3]:
        p = f"{REPO_DIR}/results/perquery/tgn_wiki_test_seed{s}.npy"
        if os.path.exists(p):
            rr = np.load(p)
            print(f"Seed {s:<3} | {rr.mean():<14.4f} | {rr[seen].mean():<16.4f} | {rr[~seen].mean():<14.4f}")
except Exception as e:
    print("Could not compute RQ2 table:", e)

# B. Review TGN Summary CSV
print("\n[tgbl-review: TGN Performance]")
review_rows = []
for s in [1, 2]:
    lp = f"{REPO_DIR}/logs/tgn_review_seed{s}.txt"
    if os.path.exists(lp):
        t = open(lp, errors="ignore").read().replace("\r", "\n")
        val_matches = re.findall(r"Validation mrr:\s*([0-9.]+)", t)
        test_matches = re.findall(r"Test: mrr:\s*([0-9.]+)", t)
        b_val = max(map(float, val_matches)) if val_matches else None
        t_mrr = float(test_matches[-1]) if test_matches else None
        if t_mrr is not None:
            review_rows.append((s, b_val, t_mrr))
            print(f"Seed {s}: Best Val MRR = {b_val:.4f} | Test MRR = {t_mrr:.4f}")

if review_rows:
    with open(f"{REPO_DIR}/results/tgn_review.csv", "w") as f:
        f.write("seed,best_val_mrr,test_mrr\n")
        for r in review_rows: f.write(f"{r[0]},{r[1]},{r[2]}\n")
    save_and_push("Update tgn_review.csv summary", ["results/tgn_review.csv"])
    print("✅ results/tgn_review.csv saved and pushed!")

⏳ Checking and installing torch_scatter...
✅ torch_scatter successfully loaded on GPU!
✅ scripts/tgn_wiki_perquery.py prepared!

📊 OVERALL PIPELINE PROGRESS: [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 0.0% COMPLETE (Elapsed: 00:00:00)
▶ Currently Running [1/4]: Wiki TGN per-query (Seed 1)

INFO: Arguments: Namespace(data='tgbl-wiki', lr=0.0001, bs=200, k_value=10, num_epoch=30, seed=1, mem_dim=100, time_dim=100, emb_dim=100, tolerance=1e-06, patience=5, num_run=1)
downloading Dataset: tgbl-wiki to /kaggle/working/Temporal-Graph-ML-Project/third_party/TGB/tgb/datasets/tgbl_wiki/tgbl-wiki.zip
=================*** TGN: LinkPropPred: tgbl-wiki ***=============
INFO: Create directory /kaggle/working/Temporal-Graph-ML-Project/scripts/saved_results
-------------------------------------------------------------------------------
INFO: >>>>> Run: 0 <<<<<
INFO: Create directory /kaggle/working/Temporal-Graph-ML-Project/scripts/saved_models/
Epoch: 01, Loss: 1.3160, Training elapsed Time (s):  13.7853
/kagg